# Failed attempts for a new system

## First parallel implementation. 

Works for both ELM and Parareal. Now just need to find the right combination of:
- Solvers
- Size of the grid
- Dimension of the grid
And compare RandNet with nnGParareal

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import os
import sys

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE

class KuramotoSivashinskyPDE(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""
        state_lap = state.laplace(bc="auto_periodic_neumann")
        state_lap2 = state_lap.laplace(bc="auto_periodic_neumann")
        state_grad = state.gradient(bc="auto_periodic_neumann")
        return -state_grad.to_scalar("squared_sum") / 2 - state_lap - state_lap2
    

class CstmConsistencyTracker(pde.ConsistencyTracker):
    def handle(self, field, t: float) -> None:
        if not np.all(np.isfinite(field.data)):
            raise Exception("Field was not finite")

class CstmController(pde.Controller):
    def run(self, initial_state, dt= None):
        # copy the initial state to not modify the supplied one
        if getattr(self.solver, "pde") and self.solver.pde.complex_valued:
            self._logger.info("Convert state to complex numbers")
            state = initial_state.copy(dtype=complex)
        else:
            state = initial_state.copy()

        # decide whether to call the main routine or whether this is an MPI client
        # this node is the primary one
        self._run_single(state, dt)

        return state
    
    def _run_single(self, state, dt: float | None = None) -> None:
        # gather basic information
        t_start, t_end = self.t_range
        get_time = self._get_current_time

        # initialize the stepper
        stepper = self.solver.make_stepper(state=state, dt=dt)
        stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")


class KSSolver(SolverAbstr):
    def __init__(self, grid, Gdt, Fdt, G, F):
        self.grid=grid
        self.Gdt = Gdt
        self.Fdt = Fdt
        self.F = F
        self.G = G

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(self.G, t_range=(t0,t1), tracker=CstmConsistencyTracker())
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(self.F, t_range=(t0,t1), tracker=CstmConsistencyTracker())
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)
    
    N = 20
    T = 30
    d = 32
    space_size = 2

    rng = np.random.default_rng(2345)
    eq = KuramotoSivashinskyPDE()
    grid = pde.UnitGrid([d]*space_size)  
    init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
    init_cond = init_cond_state.data
    ode = KSODE(d, space_size, init_cond)

    dt = 0.009
    F = pde.ExplicitSolver(eq, scheme='euler')
    G = pde.ExplicitSolver(eq, scheme='euler')
    solver = KSSolver(grid, dt, 0.001, G, F)
    p = PararealLight(ode, solver, (0, T), N, epsilon=5e-7)

    res = p.run(pool=pool, parall='mpi', light=True)
    print(res['timings'])


## Exploring the right combinations for the different parameters. 

Note: ELM is not performing well as of now, need to understand why.

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import os
import sys
from itertools import product

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import pickle

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

class KuramotoSivashinskyPDE(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""
        state_lap = state.laplace(bc="auto_periodic_neumann")
        state_lap2 = state_lap.laplace(bc="auto_periodic_neumann")
        state_grad = state.gradient(bc="auto_periodic_neumann")
        return -state_grad.to_scalar("squared_sum") / 2 - state_lap - state_lap2
    

class CstmConsistencyTracker(pde.ConsistencyTracker):
    def handle(self, field, t: float) -> None:
        if not np.all(np.isfinite(field.data)):
            raise Exception("Field was not finite")

class CstmController(pde.Controller):
    def run(self, initial_state, dt= None):
        # copy the initial state to not modify the supplied one
        if getattr(self.solver, "pde") and self.solver.pde.complex_valued:
            self._logger.info("Convert state to complex numbers")
            state = initial_state.copy(dtype=complex)
        else:
            state = initial_state.copy()

        # decide whether to call the main routine or whether this is an MPI client
        # this node is the primary one
        self._run_single(state, dt)

        return state
    
    def _run_single(self, state, dt: float | None = None) -> None:
        # gather basic information
        t_start, t_end = self.t_range
        get_time = self._get_current_time

        # initialize the stepper
        stepper = self.solver.make_stepper(state=state, dt=dt)
        stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")


class KSSolver(SolverAbstr):
    def __init__(self, grid, Gdt, Fdt, G, F):
        self.grid=grid
        self.Gdt = Gdt
        self.Fdt = Fdt
        self.F = F
        self.G = G

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(self.G, t_range=(t0,t1), tracker=CstmConsistencyTracker())
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(self.F, t_range=(t0,t1), tracker=CstmConsistencyTracker())
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        

N = 50
# T = 30
d = 32
space_size = 3

rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDE()
grid = pde.UnitGrid([d]*space_size)  
init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data
ode = KSODE(d, space_size, init_cond)

def do(inp):
    Fdt, dt, mdl, (Fs, Gs), T = inp
    F = pde.ExplicitSolver(eq, scheme=Fs)
    G = pde.ExplicitSolver(eq, scheme=Gs)
    solver = KSSolver(grid, dt, Fdt, G, F)
    p = PararealLight(ode, solver, (0, T), N, epsilon=5e-7)
    if mdl == 0:
        tag = 'para'
    else:
        tag = 'elm'
    l = ['thegood_2d',tag, T, dt,Fdt, mdl, Fs, Gs]
    try:
        if mdl == 0:
            res = p.run(verbose='')
        else:
            res = p.run(model='elm', degree=1, m=4, res_size=100, verbose='')
        k = res['k']
        l.append(k)
        l.append('')
    except Exception as e:
        err = str(e)
        l.append(-1)
        l.append(err)
    print(l)
    return l

if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)

    Gdt = [1, 0.5,0.2, 0.1, 0.07, 0.05, 0.03, 0.02, 0.01, 0.007, 0.005, 0.002, 0.001]
    Fdt = [0.001, 0.0001, 0.00001]
    F_G = [('rk', 'euler'), ('rk', 'rk')]
    T = [30, 50, 70, 100, 150, 200, 300, 400, 700, 1000]
    mdl = [0, 1]


    # import time
    # print('sleeping', time.time())
    # t = time.localtime()
    # current_time = time.strftime("%H:%M:%S", t)
    # print('current_time:', current_time)

    
    # time.sleep(30)
    # # convert time to readable format
    # t = time.localtime()
    # current_time = time.strftime("%H:%M:%S", t)
    # print('current_time:', current_time)

    args = list(product(Fdt, Gdt,mdl, F_G, T))
    print('tot runs', len(args))

    out = list(pool.map(do, args))
    print(out)

    with open('KS_first_test_4D', 'wb') as _file:
        pickle.dump(out, _file, pickle.HIGHEST_PROTOCOL)


    # # open the pickle
    # with open('KS_first_test_3D', 'rb') as _file:
    #     out = pickle.load(_file)

    # res = p.run(pool=pool, parall='mpi', light=True)
    # print(res['timings'])


## Things can be made faster if we numba-compile. 
Here it is for one-go
Note: this takes about 20sec to compile, so you would not want to have to compile again and again for different dts

In [ ]:


from mpi4py.futures import MPIPoolExecutor
import os
import sys


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

class KuramotoSivashinskyPDE(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def __init__(self, bc="auto_periodic_neumann"):
        super().__init__()
        self.bc = bc

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""
        state_lap = state.laplace(bc=self.bc)
        state_lap2 = state_lap.laplace(bc=self.bc)
        state_grad_sq = state.gradient_squared(bc=self.bc)
        return -state_grad_sq / 2 - state_lap - state_lap2

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        gradient_squared = state.grid.make_operator("gradient_squared", bc=self.bc)
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(data, t):
            return -0.5 * gradient_squared(data) - laplace(data + laplace(data))

        return pde_rhs
    
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def make_stepper(self, state, dt: float):
        fixed_stepper = self._make_fixed_stepper(state, dt)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
    

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        if kwargs.get('_run_from_int', False):
            out = self._run_from_int(*args, **kwargs)
        else:
            out = self._run(*args, **kwargs)
        return out

        

N = 20
T = 400
d = 32
space_size = 2

rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDE()
grid = pde.UnitGrid([d]*space_size)  
init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data
ode = KSODE(d, space_size, init_cond)

dt = 0.03
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

Gdt = 0.03
Fdt = 0.001
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid), dt=Fdt)
G_fixed_stepper = G.make_stepper(pde.ScalarField(grid), dt=Gdt)
def F_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps)[0]

def G_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps)[0]


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)
    
    
    

    solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper)
    p = PararealLightMod(ode, solver, (0, T), N, epsilon=5e-7)

    res = p.run(pool=pool, parall='mpi', light=True)
    print(res['timings'])

    res = p.run(model='elm', degree=1, m=4, res_size=100,pool=pool, parall='mpi', light=True)
    print(res['timings'])


    # %time solver.run_G(0, T/N, init_cond)
    # %time solver.run_G(T/N, 2*T/N, init_cond)

    # %time solver.run_F(0, T/N, init_cond)
    # %time out = solver.run_F(T/N, 2*T/N, init_cond)

    # import matplotlib.pyplot as plt
    # plt.imshow(out)


## Bad news: it looks like that KS-eq doesn't work for ELM. 

My guess is chaos. I tried in 1D for d=4 and it works, we get speed-up. But as I go to, say, 40, we do not.

In [ ]:

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

class KuramotoSivashinskyPDE(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def __init__(self, bc="auto_periodic_neumann"):
        super().__init__()
        self.bc = bc

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""
        state_lap = state.laplace(bc=self.bc)
        state_lap2 = state_lap.laplace(bc=self.bc)
        state_grad_sq = state.gradient_squared(bc=self.bc)
        return -state_grad_sq / 2 - state_lap - state_lap2

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        gradient_squared = state.grid.make_operator("gradient_squared", bc=self.bc)
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(data, t):
            return -0.5 * gradient_squared(data) - laplace(data + laplace(data))

        return pde_rhs
    
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def make_stepper(self, state, dt: float):
        fixed_stepper = self._make_fixed_stepper(state, dt)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        

d = 40
space_size = 1
Gdt = 0.05
Fdt = 0.001

rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDE()
grid = pde.UnitGrid([d]*space_size)  
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')


F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid), dt=Fdt)
G_fixed_stepper = G.make_stepper(pde.ScalarField(grid), dt=Gdt)
def F_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps)[0]

def G_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps)[0]

solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper)
init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data
ode = KSODE(d, space_size, init_cond)

N = 30
T = 100

p = PararealLight(ode, solver, (0, T), N, epsilon=5e-7)

res = p.run()
_ = p.run(model='elm', degree=1, m=4, res_size=100)

## Using ELMs and looking at how the dataset looks

In [ ]:

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

class KuramotoSivashinskyPDE(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def __init__(self, bc="auto_periodic_neumann"):
        super().__init__()
        self.bc = bc

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""
        state_lap = state.laplace(bc=self.bc)
        state_lap2 = state_lap.laplace(bc=self.bc)
        state_grad_sq = state.gradient_squared(bc=self.bc)
        out = -state_grad_sq / 2 - state_lap - state_lap2
        mn = -400
        mx = 400
        return 2*(out-mn)/(mx-mn)-1

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        gradient_squared = state.grid.make_operator("gradient_squared", bc=self.bc)
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(data, t):
            out =  -0.5 * gradient_squared(data) - laplace(data + laplace(data))
            mn = -400
            mx = 400
            return 2*(out-mn)/(mx-mn)-1

        return pde_rhs
    
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def make_stepper(self, state, dt: float):
        fixed_stepper = self._make_fixed_stepper(state, dt)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        

d = 400
space_size = 1
Gdt = 50
Fdt = 0.001

rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDE()
grid = pde.UnitGrid([d]*space_size)  
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')


F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid), dt=Fdt)
G_fixed_stepper = G.make_stepper(pde.ScalarField(grid), dt=Gdt)
def F_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps)[0]

def G_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps)[0]

solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper)
init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data
ode = KSODE(d, space_size, init_cond)


#

class PMOD(PararealLight):
    def _parareal(self, model, early_stop=None, parall='Serial', store_int=False, light=False, **kwargs):
        tspan, N, epsilon, n = self.tspan, self.N, self.epsilon, self.n
        u0 = self.u0
        solver: SolverAbstr = self.solver

        if kwargs.get('debug', False):
            print('WARNING: PararealLight does not support debug mode')
                         
        t = np.linspace(tspan[0], tspan[1], num=N+1)           
        I = 0               

        datasets = {}              
            
        parall = parall.lower()
        if parall == 'mpi':
            if 'pool' not in kwargs:
                raise Exception('MPI parallel backend requested but no pool of worker provided')
            pool = kwargs['pool']
            
        if 'verbose' in kwargs:
            verbose = kwargs['verbose']
        else:
            verbose = self.verbose

        if verbose and parall != 'serial':
            print(f'Running {model.name} with {parall} parallel backend')
            
        conv_int = []
        err = np.empty((N+1, N))
        err.fill(np.nan)
        u_curr = np.empty((N+1, *n))
        u_next = np.empty((N+1, *n))
        uG_curr = np.empty((N+1, *n))
        uG_next = np.empty((N+1, *n))
        uF_curr = np.empty((N+1, *n))
        uF_next = np.empty((N+1, *n))
        u_curr.fill(np.nan)
        u_next.fill(np.nan)
        uG_curr.fill(np.nan)
        uG_next.fill(np.nan)
        uF_curr.fill(np.nan)
        uF_next.fill(np.nan)
        x = np.zeros((0, np.prod(n)))
        D = np.zeros((0, np.prod(n)))
        G_time = 0
        F_time = 0
        F_time_serial = 0
        u_curr[0,...] = u0
        uG_curr[0,...] = u_curr[0,...]
        uF_curr[0,...] = u_curr[0,...]
        u_next[0,...] = u_curr[0,...]
        uG_next[0,...] = u_curr[0,...]
        uF_next[0,...] = u_curr[0,...]
		
        temp = u0
        for i in range(N):
            temp, temp_t = solver.run_G_timed(t[i], t[i+1], temp)
            G_time += temp_t
            uG_curr[i+1,...] = temp
        del temp, temp_t
        u_curr[:,...] = uG_curr[:,...]
        for k in range(N):
            if verbose == 'v':
                print(f'{self.ode_name} {model.name} iteration number (out of {N}): {k+1} ')
                
            s_time = time.time()
            if parall == 'mpi':
                out = list(pool.map(solver.run_F_timed, t[I:N], t[I+1:N+1], [u_curr[i,...] for i in range(I,N)]))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
                del _temp_uFs
            elif parall == 'joblib':
                out = Parallel(-1)(delayed(lambda i: solver.run_F_timed(t[i], t[i+1], u_curr[i,...]))(i) for i in range(I,N))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
            else:
                temp_t = 0
                for i in range(I, N):
                    temp, _temp_t_int = solver.run_F_timed(t[i], t[i+1], u_curr[i,...])
                    uF_curr[i+1,...] = temp
                    temp_t += _temp_t_int
                F_time_serial += temp_t/(N-I)
            F_time += time.time() - s_time
            del s_time

            uG_next[I+1,...] = uG_curr[I+1,...]
            uF_next[I+1,...] = uF_curr[I+1,...]
            u_next[I+1,...] = uF_curr[I+1,...]
            I = I + 1
            
            x_to_add = u_curr[I-1:N+1-1,...]
            D_to_add = uF_curr[I:N+1,...] - uG_curr[I:N+1,...]
            x = np.vstack([x, x_to_add.reshape(x_to_add.shape[0], np.prod(x_to_add.shape[1:]), order='C')])
            D = np.vstack([D, D_to_add.reshape(D_to_add.shape[0], np.prod(D_to_add.shape[1:]), order='C')])
            if I == N:
                if verbose == 'v':
                    print('WARNING: early stopping')
                err_old = np.nextafter(epsilon, 0)
                err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
                err[-1,k] = np.nextafter(epsilon, 0)
                break

            model.fit_timed(x, D, k=k)

            test_x = []
            test_D = []

            for i in range(I, N):
                temp, temp_t = solver.run_G_timed(t[i], t[i+1], u_next[i,...])

                # calc truth
                F_truth, _ = solver.run_F_timed(t[i], t[i+1], u_next[i,...])
                test_x.append(np.squeeze(u_next[i,...].reshape(1,-1, order='C')))
                test_D.append(np.squeeze((F_truth - temp).reshape(1,-1, order='C')))

                G_time += temp_t
                uG_next[i+1,...] = temp
                del temp, temp_t
                mdl_inpt = u_next[i,...].reshape(1,-1, order='C')
                preds = model.predict_timed(mdl_inpt, 
                                            uF_curr[i+1,...], uG_curr[i+1,...], i=i)
                preds = preds.reshape(*n, order='C')
                u_next[i+1,...] = preds + uG_next[i+1,...]

            test_x = np.array(test_x)
            test_D = np.array(test_D)
            datasets[k] = [x.copy(), D.copy(), test_x.copy(), test_D.copy()]
            a = 0
            if np.any(np.isnan(uG_next[:,...])):
                raise Exception("NaN values in initial coarse solve - increase Ng!")
            err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
            err[I,k] = 0
            u_curr[...] = u_next[...]
            uG_curr[...] = uG_next[...]
            II = I
            for p in range(II+1, N+1):
                if err[p, k] < epsilon:
                    u_next[p,...] = u_curr[p,...]
                    uG_next[p,...] = uG_curr[p,...]
                    uF_next[p,...] = uF_curr[p,...]
                    I += 1
                else:
                    break
            uF_curr[...] = uF_next[...]
            if verbose == 'v':    
                print('--> Converged:', I)
            conv_int.append(I)
            if store_int:
                raise NotImplementedError('PararealLight does not support storing intermediate results')
            if I == N:
                break
            if (early_stop is not None) and k == (early_stop-1):
                if verbose == 'v':
                    print('Early stopping due to user condition.')
                break
        debug_dict = {}
        timings = {'F_time':F_time, 'G_time': G_time, 'F_time_serial_avg': F_time_serial/(k+1)}
        timings.update(model.get_times())
        if light:
            if 'by_iter' in timings:
                timings.pop('by_iter')
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}
        else:
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}

N = 50
T = 400

p = PMOD(ode, solver, (0, T), N, epsilon=5e-7)
# _ = p.run()
res = p.run(model='elm', degree=1, m=4, res_size=100)
print(res['timings'])

#
dataset_bkp = {}
for key in res['datasets'].keys():
    dataset_bkp[key] = [res['datasets'][key][0].copy(), res['datasets'][key][1].copy(), res['datasets'][key][2].copy(), res['datasets'][key][3].copy()]

#
df = res['datasets']
k = 2
df[k][2].shape

from models import ELM_base

elm = ELM_base(40, res_size=400, degree=1, m=4, alpha=0)

x = df[k][0]
y = df[k][1]
x_te = df[k][2]
y_te = df[k][3]

mn = np.min(x)
mx = np.max(x)

# x = 2*(x-mn)/(mx-mn)-1
# x_te = 2*(x_te-mn)/(mx-mn)-1

# OOS pred
out = []
for i in range(x_te.shape[0]):
    out.append(elm.fit_predict(x, y, x_te[[i],...], k=k))
out = np.array(out)
errors = np.max((out - y_te)**2, axis=1)
# print(f'OOS max err: {np.max(errors):0.2e}')
fig,ax = plt.subplots(figsize=(4,2))
ax.plot(np.log10(errors))
# ax.plot(np.log10(base), c='red')


# IS pred

out = []
for i in range(df[k][1].shape[0]):
    out.append(elm.fit_predict(df[k][0], df[k][1], df[k][0][[i],...], k=k))
out = np.array(out)
errors = np.max((out - df[k][1])**2, axis=1)
print(f'IS max err: {np.max(errors):0.2e}')
plt.plot(np.log10(errors))

## Normalization was the key

In [ ]:
#%%

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

class KuramotoSivashinskyPDE(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def __init__(self, bc="auto_periodic_neumann"):
        super().__init__()
        self.bc = bc

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""
        state_lap = state.laplace(bc=self.bc)
        state_lap2 = state_lap.laplace(bc=self.bc)
        state_grad_sq = state.gradient_squared(bc=self.bc)
        out = -state_grad_sq / 2 - state_lap - state_lap2
        mn = -400
        mx = 400
        return 2*(out-mn)/(mx-mn)-1

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        gradient_squared = state.grid.make_operator("gradient_squared", bc=self.bc)
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(data, t):
            out =  -0.5 * gradient_squared(data) - laplace(data + laplace(data))
            mn = -400
            mx = 400
            return 2*(out-mn)/(mx-mn)-1

        return pde_rhs
    
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def make_stepper(self, state, dt: float):
        fixed_stepper = self._make_fixed_stepper(state, dt)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        

d = 40
space_size = 1
Gdt = 30
Fdt = 0.001

rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDE()
grid = pde.UnitGrid([d]*space_size)  
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')


F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid), dt=Fdt)
G_fixed_stepper = G.make_stepper(pde.ScalarField(grid), dt=Gdt)
def F_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps)[0]

def G_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps)[0]

solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper)
init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data
ode = KSODE(d, space_size, init_cond)


#%%

class PMOD(PararealLight):
    def _parareal(self, model, early_stop=None, parall='Serial', store_int=False, light=False, **kwargs):
        tspan, N, epsilon, n = self.tspan, self.N, self.epsilon, self.n
        u0 = self.u0
        solver: SolverAbstr = self.solver

        if kwargs.get('debug', False):
            print('WARNING: PararealLight does not support debug mode')
                         
        t = np.linspace(tspan[0], tspan[1], num=N+1)           
        I = 0               

        datasets = {}              
            
        parall = parall.lower()
        if parall == 'mpi':
            if 'pool' not in kwargs:
                raise Exception('MPI parallel backend requested but no pool of worker provided')
            pool = kwargs['pool']
            
        if 'verbose' in kwargs:
            verbose = kwargs['verbose']
        else:
            verbose = self.verbose

        if verbose and parall != 'serial':
            print(f'Running {model.name} with {parall} parallel backend')
            
        conv_int = []
        err = np.empty((N+1, N))
        err.fill(np.nan)
        u_curr = np.empty((N+1, *n))
        u_next = np.empty((N+1, *n))
        uG_curr = np.empty((N+1, *n))
        uG_next = np.empty((N+1, *n))
        uF_curr = np.empty((N+1, *n))
        uF_next = np.empty((N+1, *n))
        u_curr.fill(np.nan)
        u_next.fill(np.nan)
        uG_curr.fill(np.nan)
        uG_next.fill(np.nan)
        uF_curr.fill(np.nan)
        uF_next.fill(np.nan)
        x = np.zeros((0, np.prod(n)))
        D = np.zeros((0, np.prod(n)))
        G_time = 0
        F_time = 0
        F_time_serial = 0
        u_curr[0,...] = u0
        uG_curr[0,...] = u_curr[0,...]
        uF_curr[0,...] = u_curr[0,...]
        u_next[0,...] = u_curr[0,...]
        uG_next[0,...] = u_curr[0,...]
        uF_next[0,...] = u_curr[0,...]
		
        temp = u0
        for i in range(N):
            temp, temp_t = solver.run_G_timed(t[i], t[i+1], temp)
            G_time += temp_t
            uG_curr[i+1,...] = temp
        del temp, temp_t
        u_curr[:,...] = uG_curr[:,...]
        for k in range(N):
            if verbose == 'v':
                print(f'{self.ode_name} {model.name} iteration number (out of {N}): {k+1} ')
                
            s_time = time.time()
            if parall == 'mpi':
                out = list(pool.map(solver.run_F_timed, t[I:N], t[I+1:N+1], [u_curr[i,...] for i in range(I,N)]))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
                del _temp_uFs
            elif parall == 'joblib':
                out = Parallel(-1)(delayed(lambda i: solver.run_F_timed(t[i], t[i+1], u_curr[i,...]))(i) for i in range(I,N))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
            else:
                temp_t = 0
                for i in range(I, N):
                    temp, _temp_t_int = solver.run_F_timed(t[i], t[i+1], u_curr[i,...])
                    uF_curr[i+1,...] = temp
                    temp_t += _temp_t_int
                F_time_serial += temp_t/(N-I)
            F_time += time.time() - s_time
            del s_time

            uG_next[I+1,...] = uG_curr[I+1,...]
            uF_next[I+1,...] = uF_curr[I+1,...]
            u_next[I+1,...] = uF_curr[I+1,...]
            I = I + 1
            
            x_to_add = u_curr[I-1:N+1-1,...]
            D_to_add = uF_curr[I:N+1,...] - uG_curr[I:N+1,...]
            x = np.vstack([x, x_to_add.reshape(x_to_add.shape[0], np.prod(x_to_add.shape[1:]), order='C')])
            D = np.vstack([D, D_to_add.reshape(D_to_add.shape[0], np.prod(D_to_add.shape[1:]), order='C')])
            if I == N:
                if verbose == 'v':
                    print('WARNING: early stopping')
                err_old = np.nextafter(epsilon, 0)
                err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
                err[-1,k] = np.nextafter(epsilon, 0)
                break

            model.fit_timed(x, D, k=k)

            test_x = []
            test_D = []

            for i in range(I, N):
                temp, temp_t = solver.run_G_timed(t[i], t[i+1], u_next[i,...])

                # calc truth
                F_truth, _ = solver.run_F_timed(t[i], t[i+1], u_next[i,...])
                test_x.append(np.squeeze(u_next[i,...].reshape(1,-1, order='C')))
                test_D.append(np.squeeze((F_truth - temp).reshape(1,-1, order='C')))

                G_time += temp_t
                uG_next[i+1,...] = temp
                del temp, temp_t
                mdl_inpt = u_next[i,...].reshape(1,-1, order='C')
                preds = model.predict_timed(mdl_inpt, 
                                            uF_curr[i+1,...], uG_curr[i+1,...], i=i)
                preds = preds.reshape(*n, order='C')
                u_next[i+1,...] = preds + uG_next[i+1,...]

            test_x = np.array(test_x)
            test_D = np.array(test_D)
            datasets[k] = [x.copy(), D.copy(), test_x.copy(), test_D.copy()]
            a = 0
            if np.any(np.isnan(uG_next[:,...])):
                raise Exception("NaN values in initial coarse solve - increase Ng!")
            err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
            err[I,k] = 0
            u_curr[...] = u_next[...]
            uG_curr[...] = uG_next[...]
            II = I
            for p in range(II+1, N+1):
                if err[p, k] < epsilon:
                    u_next[p,...] = u_curr[p,...]
                    uG_next[p,...] = uG_curr[p,...]
                    uF_next[p,...] = uF_curr[p,...]
                    I += 1
                else:
                    break
            uF_curr[...] = uF_next[...]
            if verbose == 'v':    
                print('--> Converged:', I)
            conv_int.append(I)
            if store_int:
                raise NotImplementedError('PararealLight does not support storing intermediate results')
            if I == N:
                break
            if (early_stop is not None) and k == (early_stop-1):
                if verbose == 'v':
                    print('Early stopping due to user condition.')
                break
        debug_dict = {}
        timings = {'F_time':F_time, 'G_time': G_time, 'F_time_serial_avg': F_time_serial/(k+1)}
        timings.update(model.get_times())
        if light:
            if 'by_iter' in timings:
                timings.pop('by_iter')
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}
        else:
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}

N = 50
T = 400

p = PMOD(ode, solver, (0, T), N, epsilon=5e-7)
res = p.run(model='elm', degree=1, m=4, res_size=100)
_ = p.run()
nngp = p.run(model='nngp')
print(res['timings'])

#%% 
dataset_bkp = {}
for key in res['datasets'].keys():
    dataset_bkp[key] = [res['datasets'][key][0].copy(), res['datasets'][key][1].copy(), res['datasets'][key][2].copy(), res['datasets'][key][3].copy()]

#%%
df = res['datasets']
k = 2
df[k][2].shape

from models import ELM_base

elm = ELM_base(40, res_size=400, degree=1, m=4, alpha=0)

x = df[k][0]
y = df[k][1]
x_te = df[k][2]
y_te = df[k][3]

mn = np.min(x)
mx = np.max(x)

# x = 2*(x-mn)/(mx-mn)-1
# x_te = 2*(x_te-mn)/(mx-mn)-1

# OOS pred
out = []
for i in range(x_te.shape[0]):
    out.append(elm.fit_predict(x, y, x_te[[i],...], k=k))
out = np.array(out)
errors = np.max((out - y_te)**2, axis=1)
# print(f'OOS max err: {np.max(errors):0.2e}')
fig,ax = plt.subplots(figsize=(4,2))
ax.plot(np.log10(errors))
# ax.plot(np.log10(base), c='red')


#%% IS pred

out = []
for i in range(df[k][1].shape[0]):
    out.append(elm.fit_predict(df[k][0], df[k][1], df[k][0][[i],...], k=k))
out = np.array(out)
errors = np.max((out - df[k][1])**2, axis=1)
print(f'IS max err: {np.max(errors):0.2e}')
plt.plot(np.log10(errors))

## Cahn Hillird

In [ ]:
#%%

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

class CahnHilliardPDE(pde.PDEBase):
    r"""A simple Cahn-Hilliard equation.

    The mathematical definition is

    .. math::
        \partial_t c = \nabla^2 \left(c^3 - c - \gamma \nabla^2 c\right)

    where :math:`c` is a scalar field taking values on the interval :math:`[-1, 1]` and
    :math:`\gamma` sets the (squared) interfacial width.
    """

    explicit_time_dependence = False

    def __init__(
        self,
        interface_width: float = 1, diff=1, lambd=1,
        bc_c = "auto_periodic_neumann",
        bc_mu = "auto_periodic_neumann",
    ):
        
        super().__init__()

        self.interface_width = interface_width
        self.bc_c = bc_c
        self.bc_mu = bc_mu
        self.diff = diff
        self.lambd = lambd

    def evolution_rate(  # type: ignore
        self,
        state,
        t: float = 0,
    ):
        """Evaluate the right hand side of the PDE.

        Args:
            state (:class:`~pde.fields.ScalarField`):
                The scalar field describing the concentration distribution
            t (float):
                The current time point

        Returns:
            :class:`~pde.fields.ScalarField`:
            Scalar field describing the evolution rate of the PDE
        """
        assert isinstance(state, pde.ScalarField), "`state` must be ScalarField"
        c_laplace = state.laplace(bc=self.bc_c, label="evolution rate", args={"t": t})
        result = state**3 - state - self.interface_width * c_laplace
        return result.laplace(bc=self.bc_mu, args={"t": t})  # type: ignore



    def _make_pde_rhs_numba(  # type: ignore
        self, state
    ) :
        """Create a compiled function evaluating the right hand side of the PDE.

        Args:
            state (:class:`~pde.fields.ScalarField`):
                An example for the state defining the grid and data types

        Returns:
            A function with signature `(state_data, t)`, which can be called with an
            instance of :class:`~numpy.ndarray` of the state data and the time to
            obtained an instance of :class:`~numpy.ndarray` giving the evolution rate.
        """
        arr_type = nb.typeof(state.data)
        signature = arr_type(arr_type, nb.double)

        interface_width = self.interface_width
        diff = self.diff
        lambd = self.lambd
        laplace_c = state.grid.make_operator("laplace", bc=self.bc_c)
        laplace_mu = state.grid.make_operator("laplace", bc=self.bc_mu)

        @nb.jit(signature)
        def pde_rhs(state_data: np.ndarray, t: float):
            """Compiled helper function evaluating right hand side."""
            mu = (
                state_data**3
                - state_data
                - interface_width * laplace_c(state_data, args={"t": t})
                +lambd * state_data**5
            )
            return diff*laplace_mu(mu, args={"t": t})

        return pde_rhs  # type: ignore
    
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def make_stepper(self, state, dt: float):
        fixed_stepper = self._make_fixed_stepper(state, dt)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        

d = 64
space_size = 1
Gdt = 0.05
Fdt = 0.0002

rng = np.random.default_rng(2345)


#%%
eq = CahnHilliardPDE(diff=50, interface_width=0.3, lambd=0.5)
grid = pde.UnitGrid([d]*space_size)  
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')


F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid), dt=Fdt)

def F_wrapped_stepper(state, t_start: float, t_end: float) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps)[0]

def run_F(t0, t1, u0):
    state = pde.ScalarField(grid, u0)
    controller = CstmController(t_range=(t0,t1), stepper=F_wrapped_stepper)
    return controller.run(state).data

init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data

#% make video using Ipython
from IPython.display import clear_output

out = init_cond
outs = [out]
for i in range(20):
    scale = 1
    out = run_F(i*scale, scale*(i+1), out)
    outs.append(out)
    plt.imshow(out)
    plt.pause(0.03)
    plt.title(scale*i)
    clear_output(wait=True)

#%%

for i in range(20):
    scale = 1
    plt.imshow(outs[i])
    plt.pause(0.2)
    plt.title(scale*i)
    clear_output(wait=True)


## Fixed Norm, MPI

Fixed the normalization and implemented Explicit solver so that we can compile once and use for different dts

In [ ]:
#%%
from mpi4py.futures import MPIPoolExecutor
import os
import sys

print('starting')


from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

mn, mx = -400, 2.5

class KuramotoSivashinskyPDENorm(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def __init__(self, bc="auto_periodic_neumann"):
        super().__init__()
        self.bc = bc

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""

        # un-normalize state
        state = (state+1)/2 * (mx-mn) + mn

        # make original computation
        state_lap = state.laplace(bc=self.bc)
        state_lap2 = state_lap.laplace(bc=self.bc)
        state_grad_sq = state.gradient_squared(bc=self.bc)
        out = -state_grad_sq / 2 - state_lap - state_lap2
        # mn = -400
        # mx = 400
        # rescale output
        scale = 2/(mx-mn)
        return scale * out

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        
        gradient_squared = state.grid.make_operator("gradient_squared", bc=self.bc)
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(data, t):
            data = (data+1)/2 * (mx-mn) + mn
            out =  -0.5 * gradient_squared(data) - laplace(data + laplace(data))
            # mn = -400
            # mx = 400
            scale = 2/(mx-mn)
            return scale * out

        return pde_rhs
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _make_single_step_fixed_euler(self, state):
        if self.pde.is_sde:
            # handle stochastic version of the pde
            raise Exception('NOt implemented 1')

        else:
            # handle deterministic version of the pde
            rhs_pde = self._make_pde_rhs(state, backend=self.backend)

            def stepper(state_data: np.ndarray, t: float, dt) -> None:
                """perform a single Euler step"""
                state_data += dt * rhs_pde(state_data, t)

        return stepper

    def _make_single_step_rk45(self, state):
        if self.pde.is_sde:
            raise RuntimeError("Runge-Kutta stepper does not support stochasticity")

        # obtain functions determining how the PDE is evolved
        rhs = self._make_pde_rhs(state, backend=self.backend)

        def stepper(state_data: np.ndarray, t: float, dt) -> None:
            """compiled inner loop for speed"""
            # calculate the intermediate values in Runge-Kutta
            k1 = dt * rhs(state_data, t)
            k2 = dt * rhs(state_data + 0.5 * k1, t + 0.5 * dt)
            k3 = dt * rhs(state_data + 0.5 * k2, t + 0.5 * dt)
            k4 = dt * rhs(state_data + k3, t + dt)

            state_data += (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return stepper

    def _make_single_step_fixed_dt(self, state):
        
        if self.scheme == "euler":
            return self._make_single_step_fixed_euler(state)
        elif self.scheme in {"runge-kutta", "rk", "rk45"}:
            return self._make_single_step_rk45(state)
        else:
            raise ValueError(f"Explicit scheme `{self.scheme}` is not supported")
        
    def _make_fixed_stepper(self, state):
        single_step = self._make_single_step_fixed_dt(state)
        # print('before')

        if self._compiled:
            # print('compiling')
            sig_single_step = (nb.typeof(state.data), nb.double, nb.double)
            single_step = nb.jit(sig_single_step)(single_step)

        def fixed_stepper(
            state_data: np.ndarray, t_start: float, steps: int, dt
        ):
            """perform `steps` steps with fixed time steps"""
            modifications = 0.0
            for i in range(steps):
                # calculate the right hand side
                t = t_start + i * dt
                single_step(state_data, t, dt)

            return t + dt

        if self._compiled:
            sig_fixed = (nb.typeof(state.data), nb.double, nb.int_, nb.double)
            fixed_stepper = nb.jit(sig_fixed)(fixed_stepper)

        return fixed_stepper

    def make_stepper(self, state):
        fixed_stepper = self._make_fixed_stepper(state)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state, dt):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end, dt)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step, Gdt, Fdt):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        self.Gdt = Gdt
        self.Fdt = Fdt
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        


class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        if kwargs.get('_run_from_int', False):
            out = self._run_from_int(*args, **kwargs)
        else:
            out = self._run(*args, **kwargs)
        return out

        


d = 32
space_size = 2


rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDENorm()
grid = pde.UnitGrid([d]*space_size)  
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')


F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid))
G_fixed_stepper = G.make_stepper(pde.ScalarField(grid))
def F_wrapped_stepper(state, t_start: float, t_end: float, Fdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps, Fdt)

def G_wrapped_stepper(state, t_start: float, t_end: float, Gdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps, Gdt)



if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)
    l = []
    for Gdt in [1, 0.5,0.2, 0.1, 0.07, 0.05, 0.03, 0.02, 0.01, 0.007, 0.005, 0.002, 0.001]:
        # Gdt = 0.02
        # l.append(Gdt)
        Fdt = 0.001
        solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper, Gdt=Gdt, Fdt=Fdt)
        init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
        init_cond = init_cond_state.data
        init_cond_old = init_cond
        # normalize initial condition
        init_cond = 2*(init_cond-mn)/(mx-mn)-1
        ode = KSODE(d, space_size, init_cond)
        
        N = 50
        T = 400

        p = PararealLightMod(ode, solver, (0, T), N, epsilon=5e-7)

        try:
            res = p.run(pool=pool, parall='mpi', light=True)
            print(res['timings'])
            l.append((Gdt, 'para', res['k']))
        except Exception as e:
            l.append((Gdt, 'para', str(e)))

        try:
            res = p.run(model='elm', degree=1, m=4, res_size=100,pool=pool, parall='mpi', light=True)
            l.append((Gdt, 'elm', res['k']))
        except Exception as e:
            l.append((Gdt, 'elm', str(e)))

        print(l)
        

        
        # print(res['timings'])


    # %time solver.run_G(0, T/N, init_cond)
    # %time solver.run_G(T/N, 2*T/N, init_cond)

    # %time solver.run_F(0, T/N, init_cond)
    # %time out = solver.run_F(T/N, 2*T/N, init_cond)

    # import matplotlib.pyplot as plt
    # plt.imshow(out)

## Cahn Hilll non reciprocal failed

Noy sure why but very unstable/expensive

In [ ]:
class CahnHilliardPDE(pde.PDEBase):
    r"""A simple Cahn-Hilliard equation.

    The mathematical definition is

    .. math::
        \partial_t c = \nabla^2 \left(c^3 - c - \gamma \nabla^2 c\right)

    where :math:`c` is a scalar field taking values on the interval :math:`[-1, 1]` and
    :math:`\gamma` sets the (squared) interfacial width.
    """

    explicit_time_dependence = False

    def __init__(
        self,
        d1=1, k=0.1, d2=1,
        bc_c = "auto_periodic_neumann",
        bc_mu = "auto_periodic_neumann",
    ):
        
        super().__init__()

        self.bc = bc_c
        self.bc_mu = bc_mu
        self.d1 = d1
        self.k = k
        self.d2 = d2

    def evolution_rate(  # type: ignore
        self,
        state,
        t: float = 0,
    ):  
        u, v = state
        rhs = state.copy()
        d1, d2, k = self.d1, self.d2, self.k
        u_lapl = u.laplace(bc=self.bc, args={"t": t})
        v_lapl = v.laplace(bc=self.bc, args={"t": t})
        w = k * u_lapl
        w_lapl = w.laplace(bc=self.bc, args={"t": t})
        res1 = (d1*(3*u**2-1))*u_lapl - d2*v_lapl + w_lapl
        res2 = d2*u_lapl + 0.1*v_lapl

        rhs[0] = res1.laplace(bc=self.bc, args={"t": t})
        rhs[1] = res2.laplace(bc=self.bc, args={"t": t})
        return rhs



    # def _make_pde_rhs_numba(  # type: ignore
    #     self, state
    # ) :
    #     arr_type = nb.typeof(state.data)
    #     signature = arr_type(arr_type, nb.double)

    #     interface_width = self.interface_width
    #     diff = self.diff
    #     lambd = self.lambd
    #     laplace_c = state.grid.make_operator("laplace", bc=self.bc_c)
    #     laplace_mu = state.grid.make_operator("laplace", bc=self.bc_mu)

    #     @nb.jit(signature)
    #     def pde_rhs(state_data: np.ndarray, t: float):
    #         """Compiled helper function evaluating right hand side."""
    #         mu = (
    #             state_data**3
    #             - state_data
    #             - interface_width * laplace_c(state_data, args={"t": t})
    #             +lambd * state_data**5
    #         )
    #         return diff*laplace_mu(mu, args={"t": t})

    #     return pde_rhs  # type: ignore

# Other Referee requests

## NNs plot

### SWE

In [ ]:
from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys
import time

from pararealml.initial_condition import DiscreteInitialCondition
from pararealml.initial_value_problem import InitialValueProblem
from pararealml.operator import Operator
from pararealml import ShallowWaterEquation, Mesh, NeumannBoundaryCondition, ConstrainedProblem, GaussianInitialCondition, vectorize_bc_function
from pararealml.operators.fdm import FDMOperator, ThreePointCentralDifferenceMethod, RK4, ForwardEulerMethod


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# print(sys.argv, sys.argv[-3:])
N, mdl, dx = 235, 'elm', 0.0701
N = int(N)
dx = float(dx)

tspan = (0.0, 20.0)

if dx == 0.1:
    mul = 7
    mulF = 1
    assert N == 235
elif dx == 0.0701:
    mul = 8
    mulF = 1
    assert N == 235
elif dx == 0.05:
    mul = 14
    mulF = 1
    assert N == 235
elif dx == 0.038:
    mul = 24
    mulF = 1
    assert N == 235
else:
    raise Exception('Invalid dx val')



class PararealMLSolver(SolverAbstr):
    def __init__(self, ivp: InitialValueProblem, f: Operator, g: Operator):
        self.ivp = ivp
        self.f = f
        self.g = g
        self.dg = g.d_t
        self.df = f.d_t
        self.cp = ivp.constrained_problem
        self.vertex_oriented = f.vertex_oriented

        # required for pickling
        # Note: as long as the boundary conditions are static, it doesn't seem to matter
        self.cp._boundary_conditions = None

    def _check_time_step_size(self, t0, t1, dt):
        delta_t = (t1 - t0) 
        if not np.isclose(delta_t, dt * round(delta_t / dt)):
            raise ValueError(
                f"fine operator time step size ({dt}) must be a "
                f"divisor of sub-IVP time slice length ({delta_t})"
            )

    def run_F_full(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.df)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.f.solve(sub_ivp, False).discrete_y(self.vertex_oriented)
        return sub_y_fine

    def run_G_full(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.dg)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.g.solve(sub_ivp, False).discrete_y(self.vertex_oriented)
        return sub_y_fine
        

    def run_F(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.df)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.f.solve_fast(sub_ivp, False)
        return sub_y_fine
    
    def run_G(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.dg)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.g.solve_fast(sub_ivp, False)
        return sub_y_fine
    
class PararealMLODE(ODE):
    def __init__(self, ivp:InitialValueProblem, f):
        self.ivp = ivp
        self.f = f
        self.vertex_oriented = f.vertex_oriented
        self.name = f'SWE_{np.prod(ivp.constrained_problem.y_vertices_shape)}'

    def get_dim(self):
        return self.ivp.constrained_problem.y_shape(self.vertex_oriented)

    def get_init_cond(self):
        return self.ivp.initial_condition.discrete_y_0(self.vertex_oriented)
    

class PMOD(PararealLight):
    def _parareal(self, model, early_stop=None, parall='Serial', store_int=False, light=False, **kwargs):
        tspan, N, epsilon, n = self.tspan, self.N, self.epsilon, self.n
        u0 = self.u0
        solver: SolverAbstr = self.solver

        if kwargs.get('debug', False):
            print('WARNING: PararealLight does not support debug mode')
                         
        t = np.linspace(tspan[0], tspan[1], num=N+1)           
        I = 0               

        datasets = {}              
            
        parall = parall.lower()
        if parall == 'mpi':
            if 'pool' not in kwargs:
                raise Exception('MPI parallel backend requested but no pool of worker provided')
            pool = kwargs['pool']
            
        if 'verbose' in kwargs:
            verbose = kwargs['verbose']
        else:
            verbose = self.verbose

        if verbose and parall != 'serial':
            print(f'Running {model.name} with {parall} parallel backend')
            
        conv_int = []
        err = np.empty((N+1, N))
        err.fill(np.nan)
        u_curr = np.empty((N+1, *n))
        u_next = np.empty((N+1, *n))
        uG_curr = np.empty((N+1, *n))
        uG_next = np.empty((N+1, *n))
        uF_curr = np.empty((N+1, *n))
        uF_next = np.empty((N+1, *n))
        u_curr.fill(np.nan)
        u_next.fill(np.nan)
        uG_curr.fill(np.nan)
        uG_next.fill(np.nan)
        uF_curr.fill(np.nan)
        uF_next.fill(np.nan)
        x = np.zeros((0, np.prod(n)))
        D = np.zeros((0, np.prod(n)))
        G_time = 0
        F_time = 0
        F_time_serial = 0
        u_curr[0,...] = u0
        uG_curr[0,...] = u_curr[0,...]
        uF_curr[0,...] = u_curr[0,...]
        u_next[0,...] = u_curr[0,...]
        uG_next[0,...] = u_curr[0,...]
        uF_next[0,...] = u_curr[0,...]
		
        temp = u0
        for i in range(N):
            temp, temp_t = solver.run_G_timed(t[i], t[i+1], temp)
            G_time += temp_t
            uG_curr[i+1,...] = temp
        del temp, temp_t
        u_curr[:,...] = uG_curr[:,...]
        for k in range(N):
            if verbose == 'v':
                print(f'{self.ode_name} {model.name} iteration number (out of {N}): {k+1} ')
                
            s_time = time.time()
            if parall == 'mpi':
                out = list(pool.map(solver.run_F_timed, t[I:N], t[I+1:N+1], [u_curr[i,...] for i in range(I,N)]))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
                del _temp_uFs
            elif parall == 'joblib':
                out = Parallel(-1)(delayed(lambda i: solver.run_F_timed(t[i], t[i+1], u_curr[i,...]))(i) for i in range(I,N))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
            else:
                temp_t = 0
                for i in range(I, N):
                    temp, _temp_t_int = solver.run_F_timed(t[i], t[i+1], u_curr[i,...])
                    uF_curr[i+1,...] = temp
                    temp_t += _temp_t_int
                F_time_serial += temp_t/(N-I)
            F_time += time.time() - s_time
            del s_time

            uG_next[I+1,...] = uG_curr[I+1,...]
            uF_next[I+1,...] = uF_curr[I+1,...]
            u_next[I+1,...] = uF_curr[I+1,...]
            I = I + 1
            
            x_to_add = u_curr[I-1:N+1-1,...]
            D_to_add = uF_curr[I:N+1,...] - uG_curr[I:N+1,...]
            x = np.vstack([x, x_to_add.reshape(x_to_add.shape[0], np.prod(x_to_add.shape[1:]), order='C')])
            D = np.vstack([D, D_to_add.reshape(D_to_add.shape[0], np.prod(D_to_add.shape[1:]), order='C')])
            if I == N:
                if verbose == 'v':
                    print('WARNING: early stopping')
                err_old = np.nextafter(epsilon, 0)
                err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
                err[-1,k] = np.nextafter(epsilon, 0)
                break

            model.fit_timed(x, D, k=k)

            test_x = []
            test_D = []

            for i in range(I, N):
                temp, temp_t = solver.run_G_timed(t[i], t[i+1], u_next[i,...])

                # # calc truth
                # F_truth, _ = solver.run_F_timed(t[i], t[i+1], u_next[i,...])
                # test_x.append(np.squeeze(u_next[i,...].reshape(1,-1, order='C')))
                # test_D.append(np.squeeze((F_truth - temp).reshape(1,-1, order='C')))

                G_time += temp_t
                uG_next[i+1,...] = temp
                del temp, temp_t
                mdl_inpt = u_next[i,...].reshape(1,-1, order='C')
                preds = model.predict_timed(mdl_inpt, 
                                            uF_curr[i+1,...], uG_curr[i+1,...], i=i)
                preds = preds.reshape(*n, order='C')
                u_next[i+1,...] = preds + uG_next[i+1,...]

            # test_x = np.array(test_x)
            # test_D = np.array(test_D)
            datasets[k] = [x.copy(), '','','']
            a = 0
            if np.any(np.isnan(uG_next[:,...])):
                raise Exception("NaN values in initial coarse solve - increase Ng!")
            err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
            err[I,k] = 0
            u_curr[...] = u_next[...]
            uG_curr[...] = uG_next[...]
            II = I
            for p in range(II+1, N+1):
                if err[p, k] < epsilon:
                    u_next[p,...] = u_curr[p,...]
                    uG_next[p,...] = uG_curr[p,...]
                    uF_next[p,...] = uF_curr[p,...]
                    I += 1
                else:
                    break
            uF_curr[...] = uF_next[...]
            if verbose == 'v':    
                print('--> Converged:', I)
            conv_int.append(I)
            if store_int:
                raise NotImplementedError('PararealLight does not support storing intermediate results')
            if I == N:
                break
            if (early_stop is not None) and k == (early_stop-1):
                if verbose == 'v':
                    print('Early stopping due to user condition.')
                break
        debug_dict = {}
        timings = {'F_time':F_time, 'G_time': G_time, 'F_time_serial_avg': F_time_serial/(k+1)}
        timings.update(model.get_times())
        if light:
            if 'by_iter' in timings:
                timings.pop('by_iter')
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}
        else:
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}
    

                          

def get_ivp(d):

    diff_eq = ShallowWaterEquation(0.5)
    mesh = Mesh([(-5.0, 5.0), (0.0, 5.0)], [d, d])
    bcs = [
        (
            NeumannBoundaryCondition(
                vectorize_bc_function(lambda x, t: (0.0, None, None)),
                is_static=True,
            ),
            NeumannBoundaryCondition(
                vectorize_bc_function(lambda x, t: (0.0, None, None)),
                is_static=True,
            ),
        )
    ] * 2
    cp = ConstrainedProblem(diff_eq, mesh, bcs)
    ic = GaussianInitialCondition(
        cp,
        [(np.array([2.5, 1.25]), np.array([[0.25, 0.0], [0.0, 0.25]]))] * 3,
        [1.0, 0.0, 0.0],
    )
    ivp = InitialValueProblem(cp, tspan, ic)
    return ivp



if __name__ == '__main__':

    # mulF=1
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    ivp = get_ivp(dx)

    f = FDMOperator(RK4(), ThreePointCentralDifferenceMethod(), (20/235)/(100*mulF), tqdm=False)
    g = FDMOperator(ForwardEulerMethod(), ThreePointCentralDifferenceMethod(), (20/235)/mul, tqdm=False)
    N = 235
    ode = PararealMLODE(ivp, f)
    
    print(N, mdl, dx, ode.name)
    
    
        
    s = PararealMLSolver(ivp, f, g)
    p = PMOD(ode, s, tspan=tspan, N=N, epsilon=5e-7)

    if mdl == 'para':
        res = p.run(pool=pool, parall='mpi', light=True)
    elif mdl == 'elm':
        res = p.run(model='elm', degree=1, m=4, pool=pool, parall='mpi', light=True)
    elif mdl == 'nngp':
        res = p.run(model='nngp', pool=pool, parall='mpi', light=True, 
                    nn=20)
    elif mdl == 'gp':
        res = p.run(model='gpjax', pool=pool, parall='mpi', light=True)
    else:
        raise Exception('Unknown model type', mdl)
        
    
    print(res['timings'])

    import scipy
    import matplotlib.pyplot as plt

    df = res['datasets']
    K = 11
    i = 201
    conv_int = res['conv_int']
    idx = i - conv_int[K-1]
    if idx < 0 :
        print('change')
    else:
        idx = idx + df[K-1][0].shape[0]-1

        target = df[K][0][[idx], ...]
        d = np.zeros((N,N))
        keys = list(res['datasets'].keys())
        old_idx = 0
        for k in sorted(keys)[:K]:
            new_idx = df[k][0].shape[0]
            x = df[k][0]
            ds = scipy.spatial.distance.cdist(target, x[old_idx:new_idx,...], metric='sqeuclidean')[0,:]
            strt = N - (new_idx - old_idx)
            d[k, strt:strt+new_idx-old_idx] = ds
            old_idx = new_idx

    with open('SWE_31k_NNs_11_200', 'wb') as _file:
        pickle.dump(d, _file, pickle.HIGHEST_PROTOCOL)

    # fig, ax = plt.subplots()
    # cmap = ax.imshow(np.log10(d[:K, ...]))
    # ax.set_xlabel('Interval $i$')
    # ax.set_ylabel('Iteration $k$')
    # ax.set_title(r'SWE, $|| \boldsymbol{U}_{i}^{k} - \boldsymbol{U}_{49}^{11}||_2^2$ ')
    # plt.colorbar(cmap, orientation='horizontal')
    # fig.tight_layout()
    # fig.savefig('SWE_31k_NNs_11_49.png')

            

In [ ]:
#%%
import pickle

fontsize = 15
fs_ticks = 13

with open('SWE_31k_NNs_11_200', 'rb') as _file:
    d = pickle.load(_file)

K = 11
fig, ax = plt.subplots()
cmap = ax.imshow(np.log10(d[:K, ...]), extent=(0, 235, 11.5, 0.5), aspect='auto')
ax.set_xlabel('Interval $i$', fontsize=fontsize)
ax.set_ylabel('Iteration $k$', fontsize=fontsize)
ax.set_title(r'SWE $d=3.1e^4$'+'\n'+r' $|| \boldsymbol{U}_{i}^{k} - \boldsymbol{U}_{200}^{12}||_2^2$ - $\log_{10}$', fontsize=fontsize+3)
cbar = fig.colorbar(cmap, orientation='horizontal')
ticklabs = cbar.ax.tick_params(axis='both', labelsize=fs_ticks)
# cbar.ax.set_yticklabels(ticklabs, fontsize=fs_ticks)
ax.set_yticks(np.arange(1,12)[::-1])
ax.tick_params(axis='both', labelsize=fs_ticks)
fig.tight_layout()
fig.savefig('SWE_31k_NNs_12_200.pdf')
fig.savefig('SWE_31k_NNs_12_200.png')

fig, ax = plt.subplots()
cmap = ax.imshow(np.log10(d[:K, 190:211]), extent=(190,210,11.5, 0.5), aspect='auto')
ax.set_xlabel('Interval $i$', fontsize=fontsize)
ax.set_ylabel('Iteration $k$', fontsize=fontsize)
ax.set_title(r'SWE $d=3.1e^4$'+'\n'+r' $|| \boldsymbol{U}_{i}^{k} - \boldsymbol{U}_{200}^{12}||_2^2$ - $\log_{10}$ - Zoomed', fontsize=fontsize+3)
cbar = fig.colorbar(cmap, orientation='horizontal')
ticklabs = cbar.ax.tick_params(axis='both', labelsize=fs_ticks)

ax.set_yticks(np.arange(1,12)[::-1])
ax.set_xticks([190,195,200,205,210])
ax.tick_params(axis='both', labelsize=fs_ticks)
fig.tight_layout()
fig.savefig('SWE_31k_NNs_12_200_zoom.png')
fig.savefig('SWE_31k_NNs_12_200_zoom.pdf')


### DR

In [ ]:
#%%
from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys
import time

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# print(sys.argv, sys.argv[-3:])
N, mdl, dx = 512, 'elm', 113
N = int(N)
dx = int(dx)
d_y = dx

T = 20

if dx == 19:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    assert N == 64
elif dx == 28:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    assert N == 128
elif dx == 41:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    assert N == 256
elif dx == 77:
    mul = 1
    G = 'RK4'
    F = 'RK8'
    assert N == 512
elif dx == 113:
    mul = 2
    G = 'RK4'
    F = 'RK8'
    assert N == 512
elif dx == 164:
    mul = 4
    G = 'RK4'
    F = 'RK8'
    assert N == 512
elif dx == 235:
    mul = 8
    G = 'RK4'
    F = 'RK8'
    assert N == 512
else:
    raise Exception('Invalid dx val')


class PMOD(PararealLight):
    def _parareal(self, model, early_stop=None, parall='Serial', store_int=False, light=False, **kwargs):
        tspan, N, epsilon, n = self.tspan, self.N, self.epsilon, self.n
        u0 = self.u0
        solver: SolverAbstr = self.solver

        if kwargs.get('debug', False):
            print('WARNING: PararealLight does not support debug mode')
                         
        t = np.linspace(tspan[0], tspan[1], num=N+1)           
        I = 0               

        datasets = {}              
            
        parall = parall.lower()
        if parall == 'mpi':
            if 'pool' not in kwargs:
                raise Exception('MPI parallel backend requested but no pool of worker provided')
            pool = kwargs['pool']
            
        if 'verbose' in kwargs:
            verbose = kwargs['verbose']
        else:
            verbose = self.verbose

        if verbose and parall != 'serial':
            print(f'Running {model.name} with {parall} parallel backend')
            
        conv_int = []
        err = np.empty((N+1, N))
        err.fill(np.nan)
        u_curr = np.empty((N+1, *n))
        u_next = np.empty((N+1, *n))
        uG_curr = np.empty((N+1, *n))
        uG_next = np.empty((N+1, *n))
        uF_curr = np.empty((N+1, *n))
        uF_next = np.empty((N+1, *n))
        u_curr.fill(np.nan)
        u_next.fill(np.nan)
        uG_curr.fill(np.nan)
        uG_next.fill(np.nan)
        uF_curr.fill(np.nan)
        uF_next.fill(np.nan)
        x = np.zeros((0, np.prod(n)))
        D = np.zeros((0, np.prod(n)))
        G_time = 0
        F_time = 0
        F_time_serial = 0
        u_curr[0,...] = u0
        uG_curr[0,...] = u_curr[0,...]
        uF_curr[0,...] = u_curr[0,...]
        u_next[0,...] = u_curr[0,...]
        uG_next[0,...] = u_curr[0,...]
        uF_next[0,...] = u_curr[0,...]
		
        temp = u0
        for i in range(N):
            temp, temp_t = solver.run_G_timed(t[i], t[i+1], temp)
            G_time += temp_t
            uG_curr[i+1,...] = temp
        del temp, temp_t
        u_curr[:,...] = uG_curr[:,...]
        for k in range(N):
            if verbose == 'v':
                print(f'{self.ode_name} {model.name} iteration number (out of {N}): {k+1} ')
                
            s_time = time.time()
            if parall == 'mpi':
                out = list(pool.map(solver.run_F_timed, t[I:N], t[I+1:N+1], [u_curr[i,...] for i in range(I,N)]))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
                del _temp_uFs
            elif parall == 'joblib':
                out = Parallel(-1)(delayed(lambda i: solver.run_F_timed(t[i], t[i+1], u_curr[i,...]))(i) for i in range(I,N))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
            else:
                temp_t = 0
                for i in range(I, N):
                    temp, _temp_t_int = solver.run_F_timed(t[i], t[i+1], u_curr[i,...])
                    uF_curr[i+1,...] = temp
                    temp_t += _temp_t_int
                F_time_serial += temp_t/(N-I)
            F_time += time.time() - s_time
            del s_time

            uG_next[I+1,...] = uG_curr[I+1,...]
            uF_next[I+1,...] = uF_curr[I+1,...]
            u_next[I+1,...] = uF_curr[I+1,...]
            I = I + 1
            
            x_to_add = u_curr[I-1:N+1-1,...]
            D_to_add = uF_curr[I:N+1,...] - uG_curr[I:N+1,...]
            x = np.vstack([x, x_to_add.reshape(x_to_add.shape[0], np.prod(x_to_add.shape[1:]), order='C')])
            D = np.vstack([D, D_to_add.reshape(D_to_add.shape[0], np.prod(D_to_add.shape[1:]), order='C')])
            if I == N:
                if verbose == 'v':
                    print('WARNING: early stopping')
                err_old = np.nextafter(epsilon, 0)
                err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
                err[-1,k] = np.nextafter(epsilon, 0)
                break

            model.fit_timed(x, D, k=k)

            test_x = []
            test_D = []

            for i in range(I, N):
                temp, temp_t = solver.run_G_timed(t[i], t[i+1], u_next[i,...])

                # # calc truth
                # F_truth, _ = solver.run_F_timed(t[i], t[i+1], u_next[i,...])
                # test_x.append(np.squeeze(u_next[i,...].reshape(1,-1, order='C')))
                # test_D.append(np.squeeze((F_truth - temp).reshape(1,-1, order='C')))

                G_time += temp_t
                uG_next[i+1,...] = temp
                del temp, temp_t
                mdl_inpt = u_next[i,...].reshape(1,-1, order='C')
                preds = model.predict_timed(mdl_inpt, 
                                            uF_curr[i+1,...], uG_curr[i+1,...], i=i)
                preds = preds.reshape(*n, order='C')
                u_next[i+1,...] = preds + uG_next[i+1,...]

            # test_x = np.array(test_x)
            # test_D = np.array(test_D)
            datasets[k] = [x.copy(), '','','']
            a = 0
            if np.any(np.isnan(uG_next[:,...])):
                raise Exception("NaN values in initial coarse solve - increase Ng!")
            err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
            err[I,k] = 0
            u_curr[...] = u_next[...]
            uG_curr[...] = uG_next[...]
            II = I
            for p in range(II+1, N+1):
                if err[p, k] < epsilon:
                    u_next[p,...] = u_curr[p,...]
                    uG_next[p,...] = uG_curr[p,...]
                    uF_next[p,...] = uF_curr[p,...]
                    I += 1
                else:
                    break
            uF_curr[...] = uF_next[...]
            if verbose == 'v':    
                print('--> Converged:', I)
            conv_int.append(I)
            if store_int:
                raise NotImplementedError('PararealLight does not support storing intermediate results')
            if I == N:
                break
            if (early_stop is not None) and k == (early_stop-1):
                if verbose == 'v':
                    print('Early stopping due to user condition.')
                break
        debug_dict = {}
        timings = {'F_time':F_time, 'G_time': G_time, 'F_time_serial_avg': F_time_serial/(k+1)}
        timings.update(model.get_times())
        if light:
            if 'by_iter' in timings:
                timings.pop('by_iter')
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}
        else:
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int, 'datasets':datasets}
                          

ode = DiffReact(d_x=dx, use_jax=False, normalization='-11')

_f = ode.get_vector_field()

def f(t, u):
    return _f(t, u)


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    print(N, mdl, dx, ode.name)

    
    s = SolverScipy(f, mul, 1e2, G, F, verbose=False, use_jax=False)
    p = PMOD(ode, s, tspan=(0, T), N=N)

    if mdl == 'para':
        res = p.run(pool=pool, parall='mpi', light=True)
    elif mdl == 'elm':
        res = p.run(model='elm', degree=1, m=3, pool=pool, parall='mpi', light=True)
    elif mdl == 'nngp':
        res = p.run(model='nngp', pool=pool, parall='mpi', light=True, 
                    nn=20)
    elif mdl == 'gp':
        res = p.run(model='gpjax', pool=pool, parall='mpi', light=True)
    else:
        raise Exception('Unknown model type', mdl)
        
    print(res['timings'])

    import scipy
    import matplotlib.pyplot as plt

    df = res['datasets']
    K = 4
    i = 301
    conv_int = res['conv_int']
    idx = i - conv_int[K-1]
    if idx < 0 :
        print('change')
    else:
        idx = idx + df[K-1][0].shape[0]-1

        target = df[K][0][[idx], ...]
        d = np.zeros((N,N))
        keys = list(res['datasets'].keys())
        old_idx = 0
        for k in sorted(keys)[:K]:
            new_idx = df[k][0].shape[0]
            x = df[k][0]
            ds = scipy.spatial.distance.cdist(target, x[old_idx:new_idx,...], metric='sqeuclidean')[0,:]
            strt = N - (new_idx - old_idx)
            d[k, strt:strt+new_idx-old_idx] = ds
            old_idx = new_idx

    
    

In [ ]:
#%%
import pickle

with open('DR_25k_NNs_4_300.pkl', 'rb') as _file:
    d = pickle.load(_file)

K = 4
fontsize = 15
fs_ticks = 13

fig, ax = plt.subplots()
cmap = ax.imshow(np.log10(d[:K, ...]), extent=(0, 512, 4.5, 0.5), aspect='auto')
ax.set_xlabel('Interval $i$', fontsize=fontsize)
ax.set_ylabel('Iteration $k$', fontsize=fontsize)
ax.set_title(r'Diffusion Reaction $d=2.5e^4$, $|| \boldsymbol{U}_{i}^{k} - \boldsymbol{U}_{300}^{5}||_2^2$ - $\log_{10}$ ', fontsize=fontsize+3)
cbar = fig.colorbar(cmap, orientation='horizontal')
ticklabs = cbar.ax.tick_params(axis='both', labelsize=fs_ticks)
ax.set_yticks([1,2,3,4][::-1])
ax.tick_params(axis='both', labelsize=fs_ticks)
fig.tight_layout()
fig.savefig('DR_25k_NNs_5_300.png')
fig.savefig('DR_25k_NNs_5_300.pdf')

fig, ax = plt.subplots()
cmap = ax.imshow(np.log10(d[:K, 290:311]), extent=(290,310,4.5,0.5), aspect='auto')
ax.set_xlabel('Interval $i$', fontsize=fontsize)
ax.set_ylabel('Iteration $k$', fontsize=fontsize)
ax.set_title(r'Diffusion Reaction $d=2.5e^4$'+'\n'+r'$|| \boldsymbol{U}_{i}^{k} - \boldsymbol{U}_{300}^{5}||_2^2$ - $\log_{10}$ - Zoomed', fontsize=fontsize+3)
cbar = fig.colorbar(cmap, orientation='horizontal')
ticklabs = cbar.ax.tick_params(axis='both', labelsize=fs_ticks)
ax.set_yticks([1,2,3,4][::-1])
ax.set_xticks([290,295,300,305,310])
ax.tick_params(axis='both', labelsize=fs_ticks)
fig.tight_layout()

fig.savefig('DR_25k_NNs_5_300_zoom.png')
fig.savefig('DR_25k_NNs_5_300_zoom.pdf')

## Create Table

### Burgers

Run Parareals

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE, Burgers, BurgersFast
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys

from pararealml.initial_condition import DiscreteInitialCondition
from pararealml.initial_value_problem import InitialValueProblem
from pararealml.operator import Operator
from pararealml import ShallowWaterEquation, Mesh, NeumannBoundaryCondition, ConstrainedProblem, GaussianInitialCondition, vectorize_bc_function
from pararealml.operators.fdm import FDMOperator, ThreePointCentralDifferenceMethod, RK4, ForwardEulerMethod


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)



N = 128
dx = 128
mdl = 'elm'

Ng = 4
Nf = 4*10000
use_jax = True
ode = Burgers(d_x=dx, normalization='-11', use_jax=use_jax)

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out
    


_f0 = ode.get_vector_field()
def f(t, u):
    return _f0(t, u)


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    
    print(N, mdl, dx, ode.name)
    
        
    T = 5.9
    s = SolverRK(f, G='RK1', Ng=Ng, F='RK8', Nf=Nf, use_jax=use_jax)
    p = PararealLightMod(ode, s, (0, T), N=N, epsilon=5e-7)


    
    #####################################
    
    # run the code, storing intermediates in custom folder
    res = p.run(pool=pool, parall='mpi', light=True)
    res = p.run(model='elm', degree=1, m=3, pool=pool, parall='mpi', light=True)
    res = p.run(model='nngp', pool=pool, parall='mpi', light=True, 
                nn=18)
    
    p.store(name='revision_Burg_128_table', path=os.getcwd())
        

Run fine solver

In [ ]:

import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE, Burgers, BurgersFast
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

N = 128
dx = 128

Ng = 4
Nf = 4*10000
use_jax = True
ode = Burgers(d_x=dx, normalization='-11', use_jax=use_jax)

_f0 = ode.get_vector_field()
def f(t, u):
    return _f0(t, u)

    
T = 5.9
s = SolverRK(f, G='RK1', Ng=Ng, F='RK8', Nf=Nf, use_jax=use_jax)
p = PararealLight(ode, s, (0, T), N=N, epsilon=5e-7)

u = p.u0.copy()
l = [u]
for i in range(N):
    print(i)
    interm = s.run_F(p.t[i], p.t[i+1], u)
    l.append(interm)
    u = interm
l = np.array(l)

with open('revision_Burg_128_fine', 'wb') as _file:
    pickle.dump(l, _file, pickle.HIGHEST_PROTOCOL)
        

combine

In [ ]:
#%%

class PararealLightMod(PararealLight):
    pass

with open('revision_Burg_128_table', 'rb') as _file:
    p = pickle.load(_file)

with open('revision_Burg_128_fine', 'rb') as _file:
    fine = pickle.load(_file)

out = []
for mdl in p.runs.keys():
    sol = p.runs[mdl]['u']
    assert sol.shape == fine.shape

    err_across_int = np.max(np.abs(sol-fine), axis=1)
    plt.plot(np.log10(err_across_int), label=f'{mdl}')
    plt.title(f'Burg_{mdl}')
    plt.legend()
    out.append((mdl, f'{err_across_int.mean():0.2e}'))
print(out)

### SWE

#### Parareals

In [ ]:
from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys
import time

from pararealml.initial_condition import DiscreteInitialCondition
from pararealml.initial_value_problem import InitialValueProblem
from pararealml.operator import Operator
from pararealml import ShallowWaterEquation, Mesh, NeumannBoundaryCondition, ConstrainedProblem, GaussianInitialCondition, vectorize_bc_function
from pararealml.operators.fdm import FDMOperator, ThreePointCentralDifferenceMethod, RK4, ForwardEulerMethod


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# print(sys.argv, sys.argv[-3:])
N, mdl, dx = 235, 'elm', 0.0701
N = int(N)
dx = float(dx)

tspan = (0.0, 20.0)

if dx == 0.1:
    mul = 7
    mulF = 1
    assert N == 235
elif dx == 0.0701:
    mul = 8
    mulF = 1
    assert N == 235
elif dx == 0.05:
    mul = 14
    mulF = 1
    assert N == 235
elif dx == 0.038:
    mul = 24
    mulF = 1
    assert N == 235
else:
    raise Exception('Invalid dx val')



class PararealMLSolver(SolverAbstr):
    def __init__(self, ivp: InitialValueProblem, f: Operator, g: Operator):
        self.ivp = ivp
        self.f = f
        self.g = g
        self.dg = g.d_t
        self.df = f.d_t
        self.cp = ivp.constrained_problem
        self.vertex_oriented = f.vertex_oriented

        # required for pickling
        # Note: as long as the boundary conditions are static, it doesn't seem to matter
        self.cp._boundary_conditions = None

    def _check_time_step_size(self, t0, t1, dt):
        delta_t = (t1 - t0) 
        if not np.isclose(delta_t, dt * round(delta_t / dt)):
            raise ValueError(
                f"fine operator time step size ({dt}) must be a "
                f"divisor of sub-IVP time slice length ({delta_t})"
            )

    def run_F_full(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.df)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.f.solve(sub_ivp, False).discrete_y(self.vertex_oriented)
        return sub_y_fine

    def run_G_full(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.dg)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.g.solve(sub_ivp, False).discrete_y(self.vertex_oriented)
        return sub_y_fine
        

    def run_F(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.df)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.f.solve_fast(sub_ivp, False)
        return sub_y_fine
    
    def run_G(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.dg)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.g.solve_fast(sub_ivp, False)
        return sub_y_fine
    
class PararealMLODE(ODE):
    def __init__(self, ivp:InitialValueProblem, f):
        self.ivp = ivp
        self.f = f
        self.vertex_oriented = f.vertex_oriented
        self.name = f'SWE_{np.prod(ivp.constrained_problem.y_vertices_shape)}'

    def get_dim(self):
        return self.ivp.constrained_problem.y_shape(self.vertex_oriented)

    def get_init_cond(self):
        return self.ivp.initial_condition.discrete_y_0(self.vertex_oriented)
    


class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out
                          

def get_ivp(d):

    diff_eq = ShallowWaterEquation(0.5)
    mesh = Mesh([(-5.0, 5.0), (0.0, 5.0)], [d, d])
    bcs = [
        (
            NeumannBoundaryCondition(
                vectorize_bc_function(lambda x, t: (0.0, None, None)),
                is_static=True,
            ),
            NeumannBoundaryCondition(
                vectorize_bc_function(lambda x, t: (0.0, None, None)),
                is_static=True,
            ),
        )
    ] * 2
    cp = ConstrainedProblem(diff_eq, mesh, bcs)
    ic = GaussianInitialCondition(
        cp,
        [(np.array([2.5, 1.25]), np.array([[0.25, 0.0], [0.0, 0.25]]))] * 3,
        [1.0, 0.0, 0.0],
    )
    ivp = InitialValueProblem(cp, tspan, ic)
    return ivp



if __name__ == '__main__':

    # mulF=1
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    ivp = get_ivp(dx)

    f = FDMOperator(RK4(), ThreePointCentralDifferenceMethod(), (20/235)/(100*mulF), tqdm=False)
    g = FDMOperator(ForwardEulerMethod(), ThreePointCentralDifferenceMethod(), (20/235)/mul, tqdm=False)
    N = 235
    ode = PararealMLODE(ivp, f)
    
    print(N, mdl, dx, ode.name)
        
    s = PararealMLSolver(ivp, f, g)
    p = PararealLightMod(ode, s, tspan=tspan, N=N, epsilon=5e-7)

    res = p.run(pool=pool, parall='mpi', light=True)
    res = p.run(model='elm', degree=1, m=4, pool=pool, parall='mpi', light=True)
        
    p.store(name=f'revision_SWE_{dx}_table', path=os.getcwd())


            

#### Fine

In [ ]:
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys
import time

from pararealml.initial_condition import DiscreteInitialCondition
from pararealml.initial_value_problem import InitialValueProblem
from pararealml.operator import Operator
from pararealml import ShallowWaterEquation, Mesh, NeumannBoundaryCondition, ConstrainedProblem, GaussianInitialCondition, vectorize_bc_function
from pararealml.operators.fdm import FDMOperator, ThreePointCentralDifferenceMethod, RK4, ForwardEulerMethod


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# print(sys.argv, sys.argv[-3:])
N, mdl, dx = 235, 'elm', 0.0701
N = int(N)
dx = float(dx)

tspan = (0.0, 20.0)

if dx == 0.1:
    mul = 7
    mulF = 1
    assert N == 235
elif dx == 0.0701:
    mul = 8
    mulF = 1
    assert N == 235
elif dx == 0.05:
    mul = 14
    mulF = 1
    assert N == 235
elif dx == 0.038:
    mul = 24
    mulF = 1
    assert N == 235
else:
    raise Exception('Invalid dx val')



class PararealMLSolver(SolverAbstr):
    def __init__(self, ivp: InitialValueProblem, f: Operator, g: Operator):
        self.ivp = ivp
        self.f = f
        self.g = g
        self.dg = g.d_t
        self.df = f.d_t
        self.cp = ivp.constrained_problem
        self.vertex_oriented = f.vertex_oriented

        # required for pickling
        # Note: as long as the boundary conditions are static, it doesn't seem to matter
        self.cp._boundary_conditions = None

    def _check_time_step_size(self, t0, t1, dt):
        delta_t = (t1 - t0) 
        if not np.isclose(delta_t, dt * round(delta_t / dt)):
            raise ValueError(
                f"fine operator time step size ({dt}) must be a "
                f"divisor of sub-IVP time slice length ({delta_t})"
            )

    def run_F_full(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.df)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.f.solve(sub_ivp, False).discrete_y(self.vertex_oriented)
        return sub_y_fine

    def run_G_full(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.dg)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.g.solve(sub_ivp, False).discrete_y(self.vertex_oriented)
        return sub_y_fine
        

    def run_F(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.df)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.f.solve_fast(sub_ivp, False)
        return sub_y_fine
    
    def run_G(self, t0, t1, u0):
        self._check_time_step_size(t0, t1, self.dg)
        sub_ivp = InitialValueProblem(self.cp, (t0,t1),
                    DiscreteInitialCondition(self.cp, u0, self.vertex_oriented))
        
        sub_y_fine = self.g.solve_fast(sub_ivp, False)
        return sub_y_fine
    
class PararealMLODE(ODE):
    def __init__(self, ivp:InitialValueProblem, f):
        self.ivp = ivp
        self.f = f
        self.vertex_oriented = f.vertex_oriented
        self.name = f'SWE_{np.prod(ivp.constrained_problem.y_vertices_shape)}'

    def get_dim(self):
        return self.ivp.constrained_problem.y_shape(self.vertex_oriented)

    def get_init_cond(self):
        return self.ivp.initial_condition.discrete_y_0(self.vertex_oriented)
    

def get_ivp(d):

    diff_eq = ShallowWaterEquation(0.5)
    mesh = Mesh([(-5.0, 5.0), (0.0, 5.0)], [d, d])
    bcs = [
        (
            NeumannBoundaryCondition(
                vectorize_bc_function(lambda x, t: (0.0, None, None)),
                is_static=True,
            ),
            NeumannBoundaryCondition(
                vectorize_bc_function(lambda x, t: (0.0, None, None)),
                is_static=True,
            ),
        )
    ] * 2
    cp = ConstrainedProblem(diff_eq, mesh, bcs)
    ic = GaussianInitialCondition(
        cp,
        [(np.array([2.5, 1.25]), np.array([[0.25, 0.0], [0.0, 0.25]]))] * 3,
        [1.0, 0.0, 0.0],
    )
    ivp = InitialValueProblem(cp, tspan, ic)
    return ivp





ivp = get_ivp(dx)

f = FDMOperator(RK4(), ThreePointCentralDifferenceMethod(), (20/235)/(100*mulF), tqdm=False)
g = FDMOperator(ForwardEulerMethod(), ThreePointCentralDifferenceMethod(), (20/235)/mul, tqdm=False)
N = 235
ode = PararealMLODE(ivp, f)

print(N, mdl, dx, ode.name)
    
s = PararealMLSolver(ivp, f, g)
p = PararealLight(ode, s, tspan=tspan, N=N, epsilon=5e-7)

u = p.u0.copy()
l = [u]
for i in range(N):
    print(i)
    interm = s.run_F(p.t[i], p.t[i+1], u)
    l.append(interm)
    u = interm
l = np.array(l)

with open(f'revision_SWE_{dx}_fine', 'wb') as _file:
    pickle.dump(l, _file, pickle.HIGHEST_PROTOCOL)

            

#### Combine


In [ ]:
#%%
for dx in [0.0701, 0.05]:
    with open(f'revision_SWE_{dx}_table', 'rb') as _file:
        p = pickle.load(_file)

    with open(f'revision_SWE_{dx}_fine', 'rb') as _file:
        fine = pickle.load(_file)

    out = []
    for mdl in p.runs.keys():
        sol = p.runs[mdl]['u']
        assert sol.shape == fine.shape

        # err_across_int = np.sum(np.sqrt((sol-fine)**2), axis=(1,2,3))
        err_across_int = np.max(np.abs(sol-fine), axis=(1,2,3))
        # plt.plot(np.log10(err_across_int), label=f'{mdl}_{dx}')
        # plt.title(f'SWE_{dx}_{mdl}')
        # plt.legend()
        out.append((dx, mdl, f'{err_across_int.mean():0.2e}'))
    print(dx, '\n', out)

### DR

#### Parareals

In [ ]:
#%%
from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys
import time

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# print(sys.argv, sys.argv[-3:])
N, mdl, dx = 512, 'elm', 113
N = int(N)
dx = int(dx)
d_y = dx

T = 20

if dx == 19:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    N = 64
elif dx == 28:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    N = 128
elif dx == 41:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    N = 256
elif dx == 77:
    mul = 1
    G = 'RK4'
    F = 'RK8'
    N = 512
elif dx == 113:
    mul = 2
    G = 'RK4'
    F = 'RK8'
    N = 512
elif dx == 164:
    mul = 4
    G = 'RK4'
    F = 'RK8'
    N = 512
elif dx == 235:
    mul = 8
    G = 'RK4'
    F = 'RK8'
    N = 512
else:
    raise Exception('Invalid dx val')

ode = DiffReact(d_x=dx, use_jax=False, normalization='-11')

_f = ode.get_vector_field()

def f(t, u):
    return _f(t, u)

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    print(N, mdl, dx, ode.name)

    
    s = SolverScipy(f, mul, 1e2, G, F, verbose=False, use_jax=False)
    p = PararealLightMod(ode, s, tspan=(0, T), N=N)

    res = p.run(pool=pool, parall='mpi', light=True)
    res = p.run(model='elm', degree=1, m=4, pool=pool, parall='mpi', light=True)
    if dx == 19:
        res = p.run(model='nngp', pool=pool, parall='mpi', light=True, 
                    nn=20)
        
    p.store(name=f'revision_DR_{dx}_table', path=os.getcwd())

    

    
    

#### Fine

In [ ]:
#%%

import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys
import time

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# print(sys.argv, sys.argv[-3:])
N, mdl, dx = 512, 'elm', 113
N = int(N)
dx = int(dx)
d_y = dx

T = 20

if dx == 19:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    N = 64
elif dx == 28:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    N = 128
elif dx == 41:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    N = 256
elif dx == 77:
    mul = 1
    G = 'RK4'
    F = 'RK8'
    N = 512
elif dx == 113:
    mul = 2
    G = 'RK4'
    F = 'RK8'
    N = 512
elif dx == 164:
    mul = 4
    G = 'RK4'
    F = 'RK8'
    N = 512
elif dx == 235:
    mul = 8
    G = 'RK4'
    F = 'RK8'
    N = 512
else:
    raise Exception('Invalid dx val')

ode = DiffReact(d_x=dx, use_jax=False, normalization='-11')

_f = ode.get_vector_field()

def f(t, u):
    return _f(t, u)


s = SolverScipy(f, mul, 1e2, G, F, verbose=False, use_jax=False)
p = PararealLight(ode, s, tspan=(0, T), N=N)

    
u = p.u0.copy()
l = [u]
for i in range(N):
    print(i)
    interm = s.run_F(p.t[i], p.t[i+1], u)
    l.append(interm)
    u = interm
l = np.array(l)

with open(f'revision_DR_{dx}_fine', 'wb') as _file:
    pickle.dump(l, _file, pickle.HIGHEST_PROTOCOL)

    
    

#### Combine


In [ ]:
#%%
for dx in [19, 41, 113]:
    with open(f'revision_DR_{dx}_table', 'rb') as _file:
        p = pickle.load(_file)

    with open(f'revision_DR_{dx}_fine', 'rb') as _file:
        fine = pickle.load(_file)

    out = []
    for mdl in p.runs.keys():
        sol = p.runs[mdl]['u']
        assert sol.shape == fine.shape

        print(np.arange(fine.ndim)[1:], fine.shape)
        err_across_int = np.max(np.abs(sol-fine), axis=np.squeeze(np.arange(fine.ndim)[1:]))
        # plt.plot(np.log10(err_across_int), label=f'{mdl}_{dx}')
        # plt.title(f'SWE_{dx}_{mdl}')
        # plt.legend()
        out.append((dx, mdl, f'{err_across_int.mean():0.2e}'))
    print(dx, '\n', out)

### Brus 2D d=32

#### Fine

In [ ]:
#%%
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr




T = 35

mn_u, mx_u = 0.3, 4
mn_v, mx_v = 0.8, 5

scal = 0.01
N = 512
Fdt = 4e-5

d = 32

if d == 20:
    space_size = 3
    Gdt = 0.052
elif d == 25:
    space_size = 3
    Gdt = 0.057
elif d == 32:
    space_size = 2
    Gdt = 0.034
elif d == 64:
    space_size = 2
    Gdt = 0.033
else:
    raise Exception('Invalid dx val')
                          

class BrusselatorPDE(pde.PDEBase):
    """Brusselator with diffusive mobility."""

    def __init__(self, a=1, b=3, diffusivity=[1, 0.1], bc="auto_periodic_neumann", rng=None):
        super().__init__()
        self.a = a
        self.b = b
        self.diffusivity = diffusivity  # spatial mobility
        self.bc = bc  # boundary condition
        self.rng = rng

    def get_initial_state(self, grid):
        """Prepare a useful initial state."""
        u = pde.ScalarField(grid, self.a, label="Field $u$")
        v = self.b / self.a + 0.1 * pde.ScalarField.random_normal(grid, label="Field $v$", rng=self.rng)
        return pde.FieldCollection([u, v])

    def evolution_rate(self, state, t=0):
        """Pure python implementation of the PDE."""
        u, v = state
        u = (u+1)/2 * (mx_u-mn_u) + mn_u
        v = (v+1)/2 * (mx_v-mn_v) + mn_v
        scale_u = 2/(mx_u-mn_u)
        scale_v = 2/(mx_v-mn_v)
        rhs = state.copy()
        d0, d1 = self.diffusivity
        rhs[0] = d0 * u.laplace(self.bc) + self.a - (self.b + 1) * u + u**2 * v
        rhs[1] = d1 * v.laplace(self.bc) + self.b * u - u**2 * v
        rhs[0] = scale_u * rhs[0] * scal
        rhs[1] = scale_v * rhs[1] * scal
        return rhs

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        d0, d1 = self.diffusivity
        a, b = self.a, self.b
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(state_data, t):
            u = state_data[0]
            v = state_data[1]
            u = (u+1)/2 * (mx_u-mn_u) + mn_u
            v = (v+1)/2 * (mx_v-mn_v) + mn_v
            scale_u = 2/(mx_u-mn_u)
            scale_v = 2/(mx_v-mn_v)

            rate = np.empty_like(state_data)
            rate[0] = scale_u * (d0 * laplace(u) + a - (1 + b) * u + v * u**2) * scal
            rate[1] = scale_v * (d1 * laplace(v) + b * u - v * u**2) * scal
            return rate

        return pde_rhs
        
class CstmExplicitSolver(pde.ExplicitSolver):
    name = 'explicit_mine'
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _make_single_step_fixed_euler(self, state):
        if self.pde.is_sde:
            # handle stochastic version of the pde
            raise Exception('NOt implemented 1')

        else:
            # handle deterministic version of the pde
            rhs_pde = self._make_pde_rhs(state, backend=self.backend)

            def stepper(state_data: np.ndarray, t: float, dt) -> None:
                """perform a single Euler step"""
                state_data += dt * rhs_pde(state_data, t)

        return stepper

    def _make_single_step_rk45(self, state):
        if self.pde.is_sde:
            raise RuntimeError("Runge-Kutta stepper does not support stochasticity")

        # obtain functions determining how the PDE is evolved
        rhs = self._make_pde_rhs(state, backend=self.backend)

        def stepper(state_data: np.ndarray, t: float, dt) -> None:
            """compiled inner loop for speed"""
            # calculate the intermediate values in Runge-Kutta
            k1 = dt * rhs(state_data, t)
            k2 = dt * rhs(state_data + 0.5 * k1, t + 0.5 * dt)
            k3 = dt * rhs(state_data + 0.5 * k2, t + 0.5 * dt)
            k4 = dt * rhs(state_data + k3, t + dt)

            state_data += (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return stepper

    def _make_single_step_fixed_dt(self, state):
        
        if self.scheme == "euler":
            return self._make_single_step_fixed_euler(state)
        elif self.scheme in {"runge-kutta", "rk", "rk45"}:
            return self._make_single_step_rk45(state)
        else:
            raise ValueError(f"Explicit scheme `{self.scheme}` is not supported")
        
    def _make_fixed_stepper(self, state):
        single_step = self._make_single_step_fixed_dt(state)
        # print('before')

        if self._compiled:
            # print('compiling')
            sig_single_step = (nb.typeof(state.data), nb.double, nb.double)
            single_step = nb.jit(sig_single_step)(single_step)

        def fixed_stepper(
            state_data: np.ndarray, t_start: float, steps: int, dt
        ):
            """perform `steps` steps with fixed time steps"""
            modifications = 0.0
            for i in range(steps):
                # calculate the right hand side
                t = t_start + i * dt
                single_step(state_data, t, dt)

            return t + dt

        if self._compiled:
            sig_fixed = (nb.typeof(state.data), nb.double, nb.int_, nb.double)
            fixed_stepper = nb.jit(sig_fixed)(fixed_stepper)

        return fixed_stepper

    def make_stepper(self, state):
        fixed_stepper = self._make_fixed_stepper(state)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state, dt):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end, dt)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step, Gdt, Fdt):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        self.Gdt = Gdt
        self.Fdt = Fdt
        

    def run_G(self, t0, t1, u0):
        fields = [pde.ScalarField(self.grid, u0[i,...]) for i in range(u0.shape[0])]
        state = pde.FieldCollection(fields)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        fields = [pde.ScalarField(self.grid, u0[i,...]) for i in range(u0.shape[0])]
        state = pde.FieldCollection(fields)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'Brus'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return self.u0.shape
        


class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        if kwargs.get('_run_from_int', False):
            out = self._run_from_int(*args, **kwargs)
        else:
            out = self._run(*args, **kwargs)
        return out
    
    def _parareal(self, model, early_stop=None, parall='Serial', store_int=False, light=False, **kwargs):
        tspan, N, epsilon, n = self.tspan, self.N, self.epsilon, self.n
        u0 = self.u0
        solver: SolverAbstr = self.solver

        if kwargs.get('debug', False):
            print('WARNING: PararealLight does not support debug mode')
                         
        t = np.linspace(tspan[0], tspan[1], num=N+1)           
        I = 0                             
            
        parall = parall.lower()
        if parall == 'mpi':
            if 'pool' not in kwargs:
                raise Exception('MPI parallel backend requested but no pool of worker provided')
            pool = kwargs['pool']
            
        if 'verbose' in kwargs:
            verbose = kwargs['verbose']
        else:
            verbose = self.verbose

        if verbose and parall != 'serial':
            print(f'Running {model.name} with {parall} parallel backend')
            
        conv_int = []
        
        # u = np.empty((N+1, n, N+1))
        # uG = np.empty((N+1, n, N+1))
        # uF = np.empty((N+1, n, N+1))
        err = np.empty((N+1, N))
        # u.fill(np.nan)
        # uG.fill(np.nan)
        # uF.fill(np.nan)
        err.fill(np.nan)

        # err_old = np.empty((N+1, N))
        # err_old.fill(np.nan)

        u_curr = np.empty((N+1, *n))
        u_next = np.empty((N+1, *n))
        uG_curr = np.empty((N+1, *n))
        uG_next = np.empty((N+1, *n))
        uF_curr = np.empty((N+1, *n))
        uF_next = np.empty((N+1, *n))
        u_curr.fill(np.nan)
        u_next.fill(np.nan)
        uG_curr.fill(np.nan)
        uG_next.fill(np.nan)
        uF_curr.fill(np.nan)
        uF_next.fill(np.nan)
        
        # x_old = np.zeros((0, n))
        # D_old = np.zeros((0,n))
        x = np.zeros((0, np.prod(n)))
        D = np.zeros((0, np.prod(n)))
        
        G_time = 0
        F_time = 0
        F_time_serial = 0
        
            
        # u[0,:,:] = u0[:, np.newaxis]
        # uG[0,:,:] = u[0,:,:]
        # uF[0,:,:] = u[0,:,:]

        u_curr[0,...] = u0
        uG_curr[0,...] = u_curr[0,...]
        uF_curr[0,...] = u_curr[0,...]
        u_next[0,...] = u_curr[0,...]
        uG_next[0,...] = u_curr[0,...]
        uF_next[0,...] = u_curr[0,...]
        
        
        # Initialization: run G sequentially
        temp = u0
        for i in range(N):
            temp, temp_t = solver.run_G_timed(t[i], t[i+1], temp)
            G_time += temp_t
            # uG[i+1,:,0] = temp
            uG_curr[i+1,...] = temp

        del temp, temp_t
        # u[:,:,0] = uG[:,:,0]
        u_curr[:,...] = uG_curr[:,...]

        
        #Step 2: integrate using F (fine solver) in parallel with the current best initial
        # values
        for k in range(N):
            if verbose == 'v':
                print(f'{self.ode_name} {model.name} iteration number (out of {N}): {k+1} ')
                
            s_time = time.time()
            if parall == 'mpi':
                out = list(pool.map(solver.run_F_timed, t[I:N], t[I+1:N+1], [u_curr[i,...] for i in range(I,N)]))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
                del _temp_uFs
            elif parall == 'joblib':
                out = Parallel(-1)(delayed(lambda i: solver.run_F_timed(t[i], t[i+1], u_curr[i,...]))(i) for i in range(I,N))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
            else:
                temp_t = 0
                for i in range(I, N):
                    # temp_old = solver.run_F_timed(t[i], t[i+1], u[i,:,k])
                    temp, _temp_t_int = solver.run_F_timed(t[i], t[i+1], u_curr[i,...])
                    # uF[i+1,:,k] = temp_old[0]
                    uF_curr[i+1,...] = temp
                    temp_t += _temp_t_int
                F_time_serial += temp_t/(N-I)
            F_time += time.time() - s_time
            del s_time
            # save values forward (as solution at time I+1 is now converged)

            uG_next[I+1,...] = uG_curr[I+1,...]
            uF_next[I+1,...] = uF_curr[I+1,...]
            u_next[I+1,...] = uF_curr[I+1,...]
            I = I + 1
            # collect training data
            x_to_add = u_curr[I-1:N+1-1,...]
            D_to_add = uF_curr[I:N+1,...] - uG_curr[I:N+1,...]
            x = np.vstack([x, x_to_add.reshape(x_to_add.shape[0], np.prod(x_to_add.shape[1:]), order='C')])
            D = np.vstack([D, D_to_add.reshape(D_to_add.shape[0], np.prod(D_to_add.shape[1:]), order='C')])
            
            
            # early stop if only one interval was missing
            if I == N:
                if verbose == 'v':
                    print('WARNING: early stopping')
                err_old = np.nextafter(epsilon, 0)
                err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
                err[-1,k] = np.nextafter(epsilon, 0)
                break
            
            
            model.fit_timed(x, D, k=k)
                
            for i in range(I, N):

                

                # run G solver on best initial value
                temp, temp_t = solver.run_G_timed(t[i], t[i+1], u_next[i,...])
                G_time += temp_t
                uG_next[i+1,...] = temp
                del temp, temp_t
                
                mdl_inpt = u_next[i,...].reshape(1,-1, order='C')
                # preds = model.predict_timed(u_next[i,...].reshape(1,-1), 
                #                             uF_curr[i+1,...], uG_curr[i+1,...], i=i)

                _start_time_ = time.time()
                preds = model.predict_timed(mdl_inpt, 
                                            uF_curr[i+1,...], uG_curr[i+1,...], i=i)
                print(f'Done update {i-I} of {N-I} in {time.time()-_start_time_}s')
                preds = preds.reshape(*n, order='C')
                
                
                # do predictor-corrector update
                # u[i+1,:,k+1] = uF[i+1,:,k] + uG[i+1,:,k+1] - uG[i+1,:,k]
                u_next[i+1,...] = preds + uG_next[i+1,...]
                
            
            # error catch
            a = 0
            if np.any(np.isnan(uG_next[:,...])):
                raise Exception("NaN values in initial coarse solve - increase Ng!")
            # if np.any(np.isnan(uG[:,:, k+1])):
            #     raise Exception("NaN values in initial coarse solve - increase Ng!")
                           
            # Step 4: Converence check
            # checks whether difference between solutions at successive iterations
            # are small, if so then that time slice is considered converged. 
                          
            # err_old[:,k] = np.linalg.norm(u[:,:,k+1] - u[:,:,k], np.inf, 1)
            # err_old[I,k] = 0

            err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
            # err[:,k] = np.linalg.norm(u_next[:,...] - u_curr[:,...], np.inf, 1)
            err[I,k] = 0
            

            u_curr[...] = u_next[...]
            uG_curr[...] = uG_next[...]
            II = I
            for p in range(II+1, N+1):
                if err[p, k] < epsilon:
                    u_next[p,...] = u_curr[p,...]
                    uG_next[p,...] = uG_curr[p,...]
                    uF_next[p,...] = uF_curr[p,...]
                    I += 1
                else:
                    break
            uF_curr[...] = uF_next[...]


            if verbose == 'v':    
                print('--> Converged:', I)
            conv_int.append(I)
            if store_int:
                raise NotImplementedError('PararealLight does not support storing intermediate results')
                # name_base = f'{self.ode_name}_{self.N}_{model.name}_int'
                # int_dir = kwargs.get('int_dir', '')
                # name_base = kwargs.get('int_name', name_base)
                # int_name = f'{name_base}_{k}'
                # _objs = {'t':t, 'I':I, 'verbose':verbose,
                #      'u':u[...,:k+2], 'uG':uG[...,:k+2], 'uF':uF[...,:k+2], 'err':err[...,:k+2], 'x':x, 'D':D, 
                #      'G_time':G_time, 'F_time':F_time,
                #      'early_stop':early_stop, 'parall':parall, 'store_int':store_int, 'kwargs':kwargs,
                #      'k':k, 'conv_int': conv_int}
                
                # self.store(path=os.path.join(int_dir, name_base), name=int_name, mdl=model, objs=_objs)
                
                
            if I == N:
                break
            if (early_stop is not None) and k == (early_stop-1):
                if verbose == 'v':
                    print('Early stopping due to user condition.')
                break
        
        debug_dict = {}
            
            
        timings = {'F_time':F_time, 'G_time': G_time, 'F_time_serial_avg': F_time_serial/(k+1)}
        timings.update(model.get_times())
        
        # return {'t':t, 'u':u[:,:,:k+1], 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
        #         'u_curr':u_curr, 'uG_curr':uG_curr, 'uF_curr':uF_curr, 'err_old':err_old[:,:k+1],
        #         'uF':uF[:,:,:k+1], 'uG':uG[:,:,:k+1], 'x_old':x_old, 'D_old':D_old,
        #         'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
        #         'conv_int':conv_int}

        if light:
            if 'by_iter' in timings:
                timings.pop('by_iter')
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int}
        else:
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int}



rng = np.random.default_rng(2345)
eq = BrusselatorPDE(rng=rng)
grid = pde.UnitGrid([d]*space_size, periodic=[True]*space_size)  

state = eq.get_initial_state(grid)
init_cond = state.data
init_cond[0,...] = 2*(init_cond[0,...]-mn_u)/(mx_u-mn_u)-1
init_cond[1,...] = 2*(init_cond[1,...]-mn_v)/(mx_v-mn_v)-1

F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.FieldCollection([pde.ScalarField(grid),pde.ScalarField(grid)]))
G_fixed_stepper = G.make_stepper(pde.FieldCollection([pde.ScalarField(grid),pde.ScalarField(grid)]))
def F_wrapped_stepper(state, t_start: float, t_end: float, Fdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps, Fdt)

def G_wrapped_stepper(state, t_start: float, t_end: float, Gdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps, Gdt) 


s = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper, Gdt=Gdt, Fdt=Fdt)
ode = KSODE(d, space_size, init_cond.copy()) 
p = PararealLightMod(ode, s, (0, T), N, epsilon=5e-7)

u = p.u0.copy()
l = [u]
for i in range(N):
    print(i)
    %time interm = s.run_F(p.t[i], p.t[i+1], u)
    l.append(interm)
    u = interm
l = np.array(l)

with open('revision_Brus2D_32_fine', 'wb') as _file:
    pickle.dump(l, _file, pickle.HIGHEST_PROTOCOL)


#### Combine

In [ ]:
#%%
import os
import pickle
import numpy as np

class PararealLightMod():
    pass

with open(os.path.join('BrusPDE', f'BrusPDE_32_{mdl}'), 'rb') as _file:
    p = pickle.load(_file)

with open(f'revision_Brus2D_32_fine', 'rb') as _file:
    fine = pickle.load(_file)


for mdl in ['elm','para','nngp']:
    with open(os.path.join('BrusPDE', f'BrusPDE_32_{mdl}'), 'rb') as _file:
        p = pickle.load(_file)

    with open(f'revision_Brus2D_32_fine', 'rb') as _file:
        fine = pickle.load(_file)

    out = []
    for mdl in p.runs.keys():
        sol = p.runs[mdl]['u']
        assert sol.shape == fine.shape

        print(np.arange(fine.ndim)[1:], fine.shape)
        # err_across_int = np.max(np.abs(sol-fine), axis=np.squeeze(np.arange(fine.ndim)[1:]))
        err_across_int = np.max(np.abs(sol-fine), axis=(1,2,3))
        # plt.plot(np.log10(err_across_int), label=f'{mdl}_{dx}')
        # plt.title(f'SWE_{dx}_{mdl}')
        # plt.legend()
        out.append((mdl, f'{err_across_int.mean():0.2e}'))
    print(mdl, '\n', out)

### Result 

| PDE                         | RandNet-Parareal    | Parareal           | nnGParareal        |
| ----------------------------- | ------------------- | ------------------ | ------------------ |
| Burgers' $d=128$              | $1.06e^{-8}$ (1h 2m)     | $1.85e^{-8}$ (8h 54m) | $1.32e^{-7}$ (1h 39m) |
| Diffusion-Reaction $d=7.2e^2$ | $3.56e^{-8}$ (23m)  | $1.83e^{-8}$ (1h 40m) | $5.71e^{-7}$ (1h 11m) |
| Diffusion-Reaction $d=3.3e^3$ | $8.56e^{-10}$ (33m) | $2.45e^{-8}$ (7h 52m) | -                  |
| Diffusion-Reaction $d=2.5e^4$ | $8.09e^{-11}$ (1h 57m) | $7.43e^{-9}$ (9h 50m) | -                  |
| SWE $d=3.1e^4$                | $6.75e^{-8}$ (4h 9m)  | $5.15e^{-8}$ (15h 43m) | -                  |
| SWE $d=6.1e^4$                | $8.54e^{-9}$ (12h 34m)  | $2.84e^{-8}$ (19h 30m) | -                  |



## Poly approx of GP

### Full GP

#### Diff React

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


from models import ModelAbstr, GPjax_p, NNGP_p, ELM, ELMComplex, BareParareal
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
import scipy
import time
class PolyReg_base():
    
    def __init__(self, d, alpha=0, degree=2, m=5, interactions=False):
        self.d = d
        self.m = m
        
        self.alpha = alpha
        self.poly = PolynomialFeatures(degree=degree)
        self.poly.fit(np.zeros((1, d)))
        # self.degree = self.poly.n_output_features_
        self.degree = degree
        self.interactions = interactions
        
    def _fit(self, x, y):
        x = self._transform(x)
        mdl = Ridge(alpha=self.alpha)
        mdl.fit(x, y)
        return mdl
    
    def _transform(self, x):
        if self.interactions:
            x = self.poly.fit_transform(x)
        else:
            # poly adds the intercept, here we need to do it manually
            features = [x**(i+1) for i in range(self.degree)]
            features.append(np.ones((x.shape[0],1)))
            x = np.hstack(features)
        return x
        

    def fit(self, x, y, k):
        self.x = x
        self.y = y
        self.k = k
        

    def predict(self, new_x):
        

        if str(self.m) == 'all':
            xm=self.x.copy()
            ym=self.y.copy()
        else:
            s_idx = np.argsort(scipy.spatial.distance.cdist(new_x, self.x, metric='sqeuclidean')[0,:])
            xm = self.x[s_idx[:self.m], :]
            ym = self.y[s_idx[:self.m], :]
        
        new_X = self._transform(new_x)
        
        mdl = self._fit(xm, ym)
        preds = np.squeeze(mdl.predict(new_X))
        return preds
        
    def fit_predict(self, x, y, new_x, k):
        self.fit(x, y, k)
        return self.predict(new_x)
    
    
class PolyReg(ModelAbstr):
    def __init__(self, d, N, alpha=0, degree=2, m=4, interactions=False, **kwargs):
        super().__init__(N=N, **kwargs)
        self.polyreg = PolyReg_base(d, alpha=alpha, degree=degree, m=m, interactions=interactions)
        self.name = f'{degree}-Poly_{m}_{"interactions" if interactions else "no_interactions"}' 


    def fit(self, x, y, k, *args, **kwargs):
        self.polyreg.fit(x, y, k)


    def predict(self, new_x, *args, **kwargs):
        return self.polyreg.predict(new_x)

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out
    
    def _run(self, model='parareal', cstm_mdl_name=None, add_model=False, **kwargs):
        n = np.prod(self.n)
        if isinstance(model, ModelAbstr):
            mdl = model
        else:
            if model.lower() == 'parareal':
                mdl = BareParareal(N=self.N, **kwargs)
            elif model.lower() == 'gpjax':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = GPjax_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'nngp':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = NNGP_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'elm':
                mdl = ELM(d=n, N=self.N, **kwargs)
            elif model.lower() == 'poly':
                mdl = PolyReg(d=n, N=self.N, **kwargs)
            elif model.lower() == 'elmcomplex':
                mdl = ELMComplex(d=n, N=self.N, **kwargs)
            else:
                raise Exception('Not implemented')
        
        
        s_time = time.time()
        out = self._parareal(mdl, **kwargs)
        elap_time = time.time() - s_time
        out['timings']['runtime'] = elap_time
        if self.verbose == 'v':
            print(f'Elapsed Parareal time: {elap_time:0.2f}s')
        
        if add_model:
            out['mdl'] = mdl.store()
        if cstm_mdl_name is None:
            cstm_mdl_name = mdl.name
        self.runs[cstm_mdl_name] = out
        return out


# print(sys.argv, sys.argv[-3:])
# degree = sys.argv[-1:] 
# degree = int(degree[0])
degree = 7
dx = 19
N = 64
d_y = dx

T = 20

if dx == 19:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    assert N == 64
elif dx == 28:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    mulF = 1200e2
    assert N == 128
elif dx == 41:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    mulF = 800e2
    assert N == 256
elif dx == 77:
    mul = 1
    G = 'RK4'
    F = 'RK8'
    mulF = 300e2
    assert N == 512
elif dx == 113:
    mul = 2
    G = 'RK4'
    F = 'RK8'
    mulF = 400e2
    assert N == 512
elif dx == 164:
    mul = 4
    G = 'RK4'
    F = 'RK8'
    mulF = 500e2
    assert N == 512
elif dx == 235:
    mul = 8
    G = 'RK4'
    F = 'RK8'
    mulF = 600e2
    assert N == 512
else:
    raise Exception('Invalid dx val')
                          

ode = DiffReact(d_x=dx, use_jax=False, normalization='-11')

_f = ode.get_vector_field()

def f(t, u):
    return _f(t, u)


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)
        
    s = SolverScipy(f, mul, 1e2, G, F, verbose=False, use_jax=False)
    p = PararealLightMod(ode, s, tspan=(0, T), N=N)
    
    #####################################
    
    # run the code, storing intermediates in custom folder
    l = []
    for alpha in [1e-20, 1e-19, 1e-18, 1e-17, 1e-16, 1e-15, 1e-14, 1e-13, 1e-12, 1e-11, 1e-10]:
        try:
            res = p.run(model='poly', degree=degree, m='all', pool=pool, parall='mpi', light=True, alpha=alpha)
            l.append([alpha, degree, res['k'], res['timings']])
        except Exception as e:
            l.append([alpha, degree, -1, str(e)])
        print('New print')
        print(l)
    
        

#### Burgers

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE, Burgers, BurgersFast
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys

from pararealml.initial_condition import DiscreteInitialCondition
from pararealml.initial_value_problem import InitialValueProblem
from pararealml.operator import Operator
from pararealml import ShallowWaterEquation, Mesh, NeumannBoundaryCondition, ConstrainedProblem, GaussianInitialCondition, vectorize_bc_function
from pararealml.operators.fdm import FDMOperator, ThreePointCentralDifferenceMethod, RK4, ForwardEulerMethod


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)



N = 128
dx = 128
mdl = 'elm'

degree = 1

Ng = 4
Nf = 4*20
use_jax = True
ode = Burgers(d_x=dx, normalization='-11', use_jax=use_jax)

from models import ModelAbstr, GPjax_p, NNGP_p, ELM, ELMComplex, BareParareal
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
import scipy
import time
class PolyReg_base():
    
    def __init__(self, d, alpha=0, degree=2, m=5, interactions=False):
        self.d = d
        self.m = m
        
        self.alpha = alpha
        self.poly = PolynomialFeatures(degree=degree)
        self.poly.fit(np.zeros((1, d)))
        # self.degree = self.poly.n_output_features_
        self.degree = degree
        self.interactions = interactions
        
    def _fit(self, x, y):
        x = self._transform(x)
        mdl = Ridge(alpha=self.alpha)
        mdl.fit(x, y)
        return mdl
    
    def _transform(self, x):
        if self.interactions:
            x = self.poly.fit_transform(x)
        else:
            # poly adds the intercept, here we need to do it manually
            features = [x**(i+1) for i in range(self.degree)]
            features.append(np.ones((x.shape[0],1)))
            x = np.hstack(features)
        return x
        

    def fit(self, x, y, k):
        self.x = x
        self.y = y
        self.k = k
        

    def predict(self, new_x):
        

        if str(self.m) == 'all':
            xm=self.x.copy()
            ym=self.y.copy()
        else:
            s_idx = np.argsort(scipy.spatial.distance.cdist(new_x, self.x, metric='sqeuclidean')[0,:])
            xm = self.x[s_idx[:self.m], :]
            ym = self.y[s_idx[:self.m], :]
        
        new_X = self._transform(new_x)
        
        mdl = self._fit(xm, ym)
        preds = np.squeeze(mdl.predict(new_X))
        return preds
        
    def fit_predict(self, x, y, new_x, k):
        self.fit(x, y, k)
        return self.predict(new_x)
    
    
class PolyReg(ModelAbstr):
    def __init__(self, d, N, alpha=0, degree=2, m=4, interactions=False, **kwargs):
        super().__init__(N=N, **kwargs)
        self.polyreg = PolyReg_base(d, alpha=alpha, degree=degree, m=m, interactions=interactions)
        self.name = f'{degree}-Poly_{m}_{"interactions" if interactions else "no_interactions"}' 


    def fit(self, x, y, k, *args, **kwargs):
        self.polyreg.fit(x, y, k)


    def predict(self, new_x, *args, **kwargs):
        return self.polyreg.predict(new_x)

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out
    
    def _run(self, model='parareal', cstm_mdl_name=None, add_model=False, **kwargs):
        n = np.prod(self.n)
        if isinstance(model, ModelAbstr):
            mdl = model
        else:
            if model.lower() == 'parareal':
                mdl = BareParareal(N=self.N, **kwargs)
            elif model.lower() == 'gpjax':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = GPjax_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'nngp':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = NNGP_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'elm':
                mdl = ELM(d=n, N=self.N, **kwargs)
            elif model.lower() == 'poly':
                mdl = PolyReg(d=n, N=self.N, **kwargs)
            elif model.lower() == 'elmcomplex':
                mdl = ELMComplex(d=n, N=self.N, **kwargs)
            else:
                raise Exception('Not implemented')
        
        
        s_time = time.time()
        out = self._parareal(mdl, **kwargs)
        elap_time = time.time() - s_time
        out['timings']['runtime'] = elap_time
        if self.verbose == 'v':
            print(f'Elapsed Parareal time: {elap_time:0.2f}s')
        
        if add_model:
            out['mdl'] = mdl.store()
        if cstm_mdl_name is None:
            cstm_mdl_name = mdl.name
        self.runs[cstm_mdl_name] = out
        return out
    


_f0 = ode.get_vector_field()
def f(t, u):
    return _f0(t, u)


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    
    # print(N, mdl, dx, ode.name)
    
        
    T = 5.9
    s = SolverRK(f, G='RK1', Ng=Ng, F='RK8', Nf=Nf, use_jax=use_jax)
    p = PararealLightMod(ode, s, (0, T), N=N, epsilon=5e-7)


    # run the code, storing intermediates in custom folder
    l = []
    for alpha in [1e-20, 1e-19, 1e-18, 1e-17, 1e-16, 1e-15, 1e-14, 1e-13, 1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0]:
        try:
            res = p.run(model='poly', degree=degree, m='all', pool=pool, parall='mpi', light=True, alpha=alpha)
            l.append([alpha, degree, res['k'], res['timings']])
        except Exception as e:
            l.append([alpha, degree, -1, str(e)])
        print('New print')
        print(l)
        

### nnGP

#### Diff React

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)



from models import ModelAbstr, GPjax_p, NNGP_p, ELM, ELMComplex, BareParareal
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
import scipy
import time

class PolyReg_base():
    
    def __init__(self, d, alpha, degree, m, interactions=False):
        self.d = d
        self.m = m
        
        self.alpha = alpha
        self.poly = PolynomialFeatures(degree=degree)
        self.poly.fit(np.zeros((1, d)))
        # self.degree = self.poly.n_output_features_
        self.degree = degree
        self.interactions = interactions
        
    def _fit(self, x, y):
        x = self._transform(x)
        mdl = Ridge()
        mdl.fit(x, y)
        return mdl
    
    def _transform(self, x):
        if self.interactions:
            x = self.poly.fit_transform(x)
        else:
            # poly adds the intercept, here we need to do it manually
            features = [x**(i+1) for i in range(self.degree)]
            features.append(np.ones((x.shape[0],1)))
            x = np.hstack(features)
        return x
        

    def fit(self, x, y, k):
        self.x = x
        self.y = y
        self.k = k
        

    def predict(self, new_x):
        

        if str(self.m) == 'all':
            xm=self.x.copy()
            ym=self.y.copy()
        else:
            s_idx = np.argsort(scipy.spatial.distance.cdist(new_x, self.x, metric='sqeuclidean')[0,:])
            xm = self.x[s_idx[:self.m], :]
            ym = self.y[s_idx[:self.m], :]
        
        new_X = self._transform(new_x)
        
        mdl = self._fit(xm, ym)
        preds = np.squeeze(mdl.predict(new_X))
        return preds
        
    def fit_predict(self, x, y, new_x, k):
        self.fit(x, y, k)
        return self.predict(new_x)
    
    
class PolyReg(ModelAbstr):
    def __init__(self, d, N, alpha=0, degree=2, m=20, interactions=False, **kwargs):
        super().__init__(N=N, **kwargs)
        self.polyreg = PolyReg_base(d, alpha=alpha, degree=degree, m=m-degree, interactions=interactions)
        self.name = f'{degree}-Poly_{m}_{"interactions" if interactions else "no_interactions"}' 


    def fit(self, x, y, k, *args, **kwargs):
        self.polyreg.fit(x, y, k)


    def predict(self, new_x, *args, **kwargs):
        return self.polyreg.predict(new_x)

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out
    
    def _run(self, model='parareal', cstm_mdl_name=None, add_model=False, **kwargs):
        n = np.prod(self.n)
        if isinstance(model, ModelAbstr):
            mdl = model
        else:
            if model.lower() == 'parareal':
                mdl = BareParareal(N=self.N, **kwargs)
            elif model.lower() == 'gpjax':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = GPjax_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'nngp':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = NNGP_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'elm':
                mdl = ELM(d=n, N=self.N, **kwargs)
            elif model.lower() == 'poly':
                mdl = PolyReg(d=n, N=self.N, **kwargs)
            elif model.lower() == 'elmcomplex':
                mdl = ELMComplex(d=n, N=self.N, **kwargs)
            else:
                raise Exception('Not implemented')
        
        
        s_time = time.time()
        out = self._parareal(mdl, **kwargs)
        elap_time = time.time() - s_time
        out['timings']['runtime'] = elap_time
        if self.verbose == 'v':
            print(f'Elapsed Parareal time: {elap_time:0.2f}s')
        
        if add_model:
            out['mdl'] = mdl.store()
        if cstm_mdl_name is None:
            cstm_mdl_name = mdl.name
        self.runs[cstm_mdl_name] = out
        return out


# print(sys.argv, sys.argv[-3:])
# degree = sys.argv[-1:] 
# degree = int(degree[0])
degree = 7
dx = 19
N = 64
d_y = dx

T = 20

if dx == 19:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    assert N == 64
elif dx == 28:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    mulF = 1200e2
    assert N == 128
elif dx == 41:
    mul = 1
    G = 'RK1'
    F = 'RK4'
    mulF = 800e2
    assert N == 256
elif dx == 77:
    mul = 1
    G = 'RK4'
    F = 'RK8'
    mulF = 300e2
    assert N == 512
elif dx == 113:
    mul = 2
    G = 'RK4'
    F = 'RK8'
    mulF = 400e2
    assert N == 512
elif dx == 164:
    mul = 4
    G = 'RK4'
    F = 'RK8'
    mulF = 500e2
    assert N == 512
elif dx == 235:
    mul = 8
    G = 'RK4'
    F = 'RK8'
    mulF = 600e2
    assert N == 512
else:
    raise Exception('Invalid dx val')
                          

ode = DiffReact(d_x=dx, use_jax=False, normalization='-11')

_f = ode.get_vector_field()

def f(t, u):
    return _f(t, u)


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)
        
    s = SolverScipy(f, mul, 1e2, G, F, verbose=False, use_jax=False)
    p = PararealLightMod(ode, s, tspan=(0, T), N=N)
    
    #####################################
    
    # run the code, storing intermediates in custom folder
    l = []
    alpha = 0
    
    try:
        res = p.run(model='poly', degree=degree, m=20, pool=pool, parall='mpi', light=True, alpha=alpha)
        l.append([alpha, degree, res['k'], res['timings']])
    except Exception as e:
        l.append([alpha, degree, -1, str(e)])
    print('New print')
    print(l)
    
        

#### Burgers

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact, ODE, Burgers, BurgersFast
from solver import SolverRK, SolverScipy, SolverAbstr
from parareal import PararealLight
import numpy as np
import os
import sys

from pararealml.initial_condition import DiscreteInitialCondition
from pararealml.initial_value_problem import InitialValueProblem
from pararealml.operator import Operator
from pararealml import ShallowWaterEquation, Mesh, NeumannBoundaryCondition, ConstrainedProblem, GaussianInitialCondition, vectorize_bc_function
from pararealml.operators.fdm import FDMOperator, ThreePointCentralDifferenceMethod, RK4, ForwardEulerMethod


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)



N = 128
dx = 128
mdl = 'elm'

degree = 1

Ng = 4
Nf = 4*20
use_jax = True
ode = Burgers(d_x=dx, normalization='-11', use_jax=use_jax)

from models import ModelAbstr, GPjax_p, NNGP_p, ELM, ELMComplex, BareParareal
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
import scipy
import time

class PolyReg_base():
    
    def __init__(self, d, alpha, degree, m, interactions=False):
        self.d = d
        self.m = m
        
        self.alpha = alpha
        self.poly = PolynomialFeatures(degree=degree)
        self.poly.fit(np.zeros((1, d)))
        # self.degree = self.poly.n_output_features_
        self.degree = degree
        self.interactions = interactions
        
    def _fit(self, x, y):
        x = self._transform(x)
        mdl = Ridge()
        mdl.fit(x, y)
        return mdl
    
    def _transform(self, x):
        if self.interactions:
            x = self.poly.fit_transform(x)
        else:
            # poly adds the intercept, here we need to do it manually
            features = [x**(i+1) for i in range(self.degree)]
            features.append(np.ones((x.shape[0],1)))
            x = np.hstack(features)
        return x
        

    def fit(self, x, y, k):
        self.x = x
        self.y = y
        self.k = k
        

    def predict(self, new_x):
        

        if str(self.m) == 'all':
            xm=self.x.copy()
            ym=self.y.copy()
        else:
            s_idx = np.argsort(scipy.spatial.distance.cdist(new_x, self.x, metric='sqeuclidean')[0,:])
            xm = self.x[s_idx[:self.m], :]
            ym = self.y[s_idx[:self.m], :]
        
        new_X = self._transform(new_x)
        
        mdl = self._fit(xm, ym)
        preds = np.squeeze(mdl.predict(new_X))
        return preds
        
    def fit_predict(self, x, y, new_x, k):
        self.fit(x, y, k)
        return self.predict(new_x)
    
    
class PolyReg(ModelAbstr):
    def __init__(self, d, N, alpha=0, degree=2, m=20, interactions=False, **kwargs):
        super().__init__(N=N, **kwargs)
        self.polyreg = PolyReg_base(d, alpha=alpha, degree=degree, m=m-degree, interactions=interactions)
        self.name = f'{degree}-Poly_{m}_{"interactions" if interactions else "no_interactions"}' 


    def fit(self, x, y, k, *args, **kwargs):
        self.polyreg.fit(x, y, k)


    def predict(self, new_x, *args, **kwargs):
        return self.polyreg.predict(new_x)

class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        try:
            if kwargs.get('_run_from_int', False):
                out = self._run_from_int(*args, **kwargs)
            else:
                out = self._run(*args, **kwargs)
        except Exception as e:
            raise
            
        return out
    
    def _run(self, model='parareal', cstm_mdl_name=None, add_model=False, **kwargs):
        n = np.prod(self.n)
        if isinstance(model, ModelAbstr):
            mdl = model
        else:
            if model.lower() == 'parareal':
                mdl = BareParareal(N=self.N, **kwargs)
            elif model.lower() == 'gpjax':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = GPjax_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'nngp':
                if 'pool' not in kwargs:
                    raise Exception('A worker pool must be provided to run NNGP in parallel')
                mdl = NNGP_p(n=n, N=self.N, worker_pool=kwargs['pool'], **kwargs)
            elif model.lower() == 'elm':
                mdl = ELM(d=n, N=self.N, **kwargs)
            elif model.lower() == 'poly':
                mdl = PolyReg(d=n, N=self.N, **kwargs)
            elif model.lower() == 'elmcomplex':
                mdl = ELMComplex(d=n, N=self.N, **kwargs)
            else:
                raise Exception('Not implemented')
        
        
        s_time = time.time()
        out = self._parareal(mdl, **kwargs)
        elap_time = time.time() - s_time
        out['timings']['runtime'] = elap_time
        if self.verbose == 'v':
            print(f'Elapsed Parareal time: {elap_time:0.2f}s')
        
        if add_model:
            out['mdl'] = mdl.store()
        if cstm_mdl_name is None:
            cstm_mdl_name = mdl.name
        self.runs[cstm_mdl_name] = out
        return out
    


_f0 = ode.get_vector_field()
def f(t, u):
    return _f0(t, u)


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    
    # print(N, mdl, dx, ode.name)
    
        
    T = 5.9
    s = SolverRK(f, G='RK1', Ng=Ng, F='RK8', Nf=Nf, use_jax=use_jax)
    p = PararealLightMod(ode, s, (0, T), N=N, epsilon=5e-7)


    # run the code, storing intermediates in custom folder
    l = []
    alpha = 0
    
    try:
        res = p.run(model='poly', degree=degree, m=20, pool=pool, parall='mpi', light=True, alpha=alpha)
        l.append([alpha, degree, res['k'], res['timings']])
    except Exception as e:
        l.append([alpha, degree, -1, str(e)])
    print('New print')
    print(l)

# Brusselator

In [ ]:
#%%
from mpi4py.futures import MPIPoolExecutor
import os
import sys

print('starting')


from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

mn_u, mx_u = 0.3, 4
mn_v, mx_v = 0.8, 5

scal = 0.01
d = 20
space_size = 3

N = 235

class BrusselatorPDE(pde.PDEBase):
    """Brusselator with diffusive mobility."""

    def __init__(self, a=1, b=3, diffusivity=[1, 0.1], bc="auto_periodic_neumann", rng=None):
        super().__init__()
        self.a = a
        self.b = b
        self.diffusivity = diffusivity  # spatial mobility
        self.bc = bc  # boundary condition
        self.rng = rng

    def get_initial_state(self, grid):
        """Prepare a useful initial state."""
        u = pde.ScalarField(grid, self.a, label="Field $u$")
        v = self.b / self.a + 0.1 * pde.ScalarField.random_normal(grid, label="Field $v$", rng=self.rng)
        return pde.FieldCollection([u, v])

    def evolution_rate(self, state, t=0):
        """Pure python implementation of the PDE."""
        u, v = state
        u = (u+1)/2 * (mx_u-mn_u) + mn_u
        v = (v+1)/2 * (mx_v-mn_v) + mn_v
        scale_u = 2/(mx_u-mn_u)
        scale_v = 2/(mx_v-mn_v)
        rhs = state.copy()
        d0, d1 = self.diffusivity
        rhs[0] = d0 * u.laplace(self.bc) + self.a - (self.b + 1) * u + u**2 * v
        rhs[1] = d1 * v.laplace(self.bc) + self.b * u - u**2 * v
        rhs[0] = scale_u * rhs[0] * scal
        rhs[1] = scale_v * rhs[1] * scal
        return rhs

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        d0, d1 = self.diffusivity
        a, b = self.a, self.b
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(state_data, t):
            u = state_data[0]
            v = state_data[1]
            u = (u+1)/2 * (mx_u-mn_u) + mn_u
            v = (v+1)/2 * (mx_v-mn_v) + mn_v
            scale_u = 2/(mx_u-mn_u)
            scale_v = 2/(mx_v-mn_v)

            rate = np.empty_like(state_data)
            rate[0] = scale_u * (d0 * laplace(u) + a - (1 + b) * u + v * u**2) * scal
            rate[1] = scale_v * (d1 * laplace(v) + b * u - v * u**2) * scal
            return rate

        return pde_rhs
        
class CstmExplicitSolver(pde.ExplicitSolver):
    name = 'explicit_mine'
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _make_single_step_fixed_euler(self, state):
        if self.pde.is_sde:
            # handle stochastic version of the pde
            raise Exception('NOt implemented 1')

        else:
            # handle deterministic version of the pde
            rhs_pde = self._make_pde_rhs(state, backend=self.backend)

            def stepper(state_data: np.ndarray, t: float, dt) -> None:
                """perform a single Euler step"""
                state_data += dt * rhs_pde(state_data, t)

        return stepper

    def _make_single_step_rk45(self, state):
        if self.pde.is_sde:
            raise RuntimeError("Runge-Kutta stepper does not support stochasticity")

        # obtain functions determining how the PDE is evolved
        rhs = self._make_pde_rhs(state, backend=self.backend)

        def stepper(state_data: np.ndarray, t: float, dt) -> None:
            """compiled inner loop for speed"""
            # calculate the intermediate values in Runge-Kutta
            k1 = dt * rhs(state_data, t)
            k2 = dt * rhs(state_data + 0.5 * k1, t + 0.5 * dt)
            k3 = dt * rhs(state_data + 0.5 * k2, t + 0.5 * dt)
            k4 = dt * rhs(state_data + k3, t + dt)

            state_data += (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return stepper

    def _make_single_step_fixed_dt(self, state):
        
        if self.scheme == "euler":
            return self._make_single_step_fixed_euler(state)
        elif self.scheme in {"runge-kutta", "rk", "rk45"}:
            return self._make_single_step_rk45(state)
        else:
            raise ValueError(f"Explicit scheme `{self.scheme}` is not supported")
        
    def _make_fixed_stepper(self, state):
        single_step = self._make_single_step_fixed_dt(state)
        # print('before')

        if self._compiled:
            # print('compiling')
            sig_single_step = (nb.typeof(state.data), nb.double, nb.double)
            single_step = nb.jit(sig_single_step)(single_step)

        def fixed_stepper(
            state_data: np.ndarray, t_start: float, steps: int, dt
        ):
            """perform `steps` steps with fixed time steps"""
            modifications = 0.0
            for i in range(steps):
                # calculate the right hand side
                t = t_start + i * dt
                single_step(state_data, t, dt)

            return t + dt

        if self._compiled:
            sig_fixed = (nb.typeof(state.data), nb.double, nb.int_, nb.double)
            fixed_stepper = nb.jit(sig_fixed)(fixed_stepper)

        return fixed_stepper

    def make_stepper(self, state):
        fixed_stepper = self._make_fixed_stepper(state)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state, dt):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end, dt)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step, Gdt, Fdt):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        self.Gdt = Gdt
        self.Fdt = Fdt
        

    def run_G(self, t0, t1, u0):
        fields = [pde.ScalarField(self.grid, u0[i,...]) for i in range(u0.shape[0])]
        state = pde.FieldCollection(fields)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        fields = [pde.ScalarField(self.grid, u0[i,...]) for i in range(u0.shape[0])]
        state = pde.FieldCollection(fields)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'Brus'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return self.u0.shape
        


class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        if kwargs.get('_run_from_int', False):
            out = self._run_from_int(*args, **kwargs)
        else:
            out = self._run(*args, **kwargs)
        return out

        





rng = np.random.default_rng(2345)
eq = BrusselatorPDE(rng=rng)
grid = pde.UnitGrid([d]*space_size, periodic=[True]*space_size)  

state = eq.get_initial_state(grid)
init_cond = state.data
init_cond[0,...] = 2*(init_cond[0,...]-mn_u)/(mx_u-mn_u)-1
init_cond[1,...] = 2*(init_cond[1,...]-mn_v)/(mx_v-mn_v)-1

F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.FieldCollection([pde.ScalarField(grid),pde.ScalarField(grid)]))
G_fixed_stepper = G.make_stepper(pde.FieldCollection([pde.ScalarField(grid),pde.ScalarField(grid)]))
def F_wrapped_stepper(state, t_start: float, t_end: float, Fdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps, Fdt)

def G_wrapped_stepper(state, t_start: float, t_end: float, Gdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps, Gdt)

#%

import json

def store_l(l):
    # return None
    try:
        with open('revision_Brus3.txt', 'a') as f:
            f.write(json.dumps(l)+'\n')
    except Exception as e:
        pass

def do(pool, inp):
    Gdt, T = inp
    Fdt = 0.001
    solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper, Gdt=Gdt, Fdt=Fdt)

    ode = KSODE(d, space_size, init_cond.copy()) 
    
    p = PararealLightMod(ode, solver, (0, T), N, epsilon=5e-7)

    l = [scal, d, space_size, Gdt, T, N, Fdt]
    try:
        res_elm = p.run(model='elm', degree=1, m=4,pool=pool, parall='mpi', light=True)
        l.extend(('elm', res_elm['k']))
    except Exception as e:
        l.extend(('elm', -1, str(e)))
        print(l)
        store_l(l)  
        return

    try:
        res = p.run(pool=pool, parall='mpi', light=True)
        l.extend(['para', res['k'], res_elm['timings'], res['timings']])
        print(l)
        store_l(l)
    except Exception as e:
        l.extend(['para', -1, str(e), res_elm['timings']])
        store_l(l)
        print(l)
        return
        


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    from itertools import product, repeat

    T = [20, 21, 22,25, 30, 35, 40, 45, 50, 55, 60,61,62, 65,70, 100,101,102,105, 150, 155, 200,201,202, 250, 300]
    Gdt = 10**(np.linspace(0,-3, 50))
    args = list(product(Gdt, T))

    print(len(args))

    np.random.seed(34)
    args = np.random.permutation(args)

    out = list(map(do, repeat(pool), args))


# from IPython.display import clear_output
# from matplotlib import pyplot as plt

# l = [init_cond.copy()]
# for i in range(20):
#     l.append(solver.run_F(i,i+1,l[-1]))
#     plt.imshow(l[-1][0,...])
#     plt.title(i)
#     clear_output(wait=True)
#     plt.pause(0.1)

## Combinations

In [ ]:
[0.01, 25, 3, 0.057, 35, 512, 1e-06, 'elm', 2, 'para', 9, {'F_time': 897.1429488658905, 'G_time': 2.163054943084717, 'F_time_serial_avg': 141.74074414208962, 'mdl_train_t': 0.0007197856903076172, 'mdl_pred_t': 29.64099407196045, 'mdl_tot_t': 29.641713857650757, 'runtime': 930.0171630382538}, {'F_time': 2814.73738861084, 'G_time': 4.836652517318726, 'F_time_serial_avg': 140.39700745199437, 'mdl_train_t': 4.839897155761719e-05, 'mdl_pred_t': 0.17366766929626465, 'mdl_tot_t': 0.17371606826782227, 'runtime': 2825.3765108585358}]

[0.01, 20, 3, 0.052, 35, 512, 0.0001, 'elm', 2, 'para', 8, {'F_time': 70.80837917327881, 'G_time': 1.346571922302246, 'F_time_serial_avg': 0.7147525470187148, 'mdl_train_t': 0.0004451274871826172, 'mdl_pred_t': 13.216968297958374, 'mdl_tot_t': 13.217413425445557, 'runtime': 85.93178820610046}, {'F_time': 13.52621865272522, 'G_time': 2.7567999362945557, 'F_time_serial_avg': 0.7145204296994594, 'mdl_train_t': 4.267692565917969e-05, 'mdl_pred_t': 0.09527993202209473, 'mdl_tot_t': 0.0953226089477539, 'runtime': 18.854674816131592}]

[0.01, 64, 2, 0.033, 35, 512, 0.0001, 'elm', 2, 'para', 7, {'F_time': 1.3288421630859375, 'G_time': 0.8259270191192627, 'F_time_serial_avg': 0.21217204951508056, 'mdl_train_t': 0.0002796649932861328, 'mdl_pred_t': 7.856361150741577, 'mdl_tot_t': 7.856640815734863, 'runtime': 10.316368818283081}, {'F_time': 3.767850637435913, 'G_time': 1.386918306350708, 'F_time_serial_avg': 0.21215260034265795, 'mdl_train_t': 3.504753112792969e-05, 'mdl_pred_t': 0.04156851768493652, 'mdl_tot_t': 0.04160356521606445, 'runtime': 6.252989292144775}]

[0.01, 32, 2, 0.034, 35, 512, 0.0001, 'elm', 2, 'para', 7, {'F_time': 1.076892375946045, 'G_time': 0.4872725009918213, 'F_time_serial_avg': 0.05932751753086811, 'mdl_train_t': 1.3828277587890625e-05, 'mdl_pred_t': 2.822906732559204, 'mdl_tot_t': 2.822920560836792, 'runtime': 4.483842611312866}, {'F_time': 2.9285991191864014, 'G_time': 0.7997243404388428, 'F_time_serial_avg': 0.05931247639842069, 'mdl_train_t': 2.6226043701171875e-05, 'mdl_pred_t': 0.01599407196044922, 'mdl_tot_t': 0.01602029800415039, 'runtime': 4.040657997131348}]`

## Run Scal

In [ ]:

from mpi4py.futures import MPIPoolExecutor
import pickle
from configs import Config
from systems import FHN_PDE, DiffReact
from solver import SolverRK, SolverScipy
from parareal import PararealLight
import numpy as np
import os
import sys

from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr


# print(sys.argv, sys.argv[-3:])
mdl, d = sys.argv[-2:] 
d = int(d)

T = 35

mn_u, mx_u = 0.3, 4
mn_v, mx_v = 0.8, 5

scal = 0.01
N = 512
Fdt = 1e-7


if d == 20:
    space_size = 3
    Gdt = 0.052
elif d == 25:
    space_size = 3
    Gdt = 0.057
elif d == 32:
    space_size = 2
    Gdt = 0.034
elif d == 64:
    space_size = 2
    Gdt = 0.033
else:
    raise Exception('Invalid dx val')
                          

class BrusselatorPDE(pde.PDEBase):
    """Brusselator with diffusive mobility."""

    def __init__(self, a=1, b=3, diffusivity=[1, 0.1], bc="auto_periodic_neumann", rng=None):
        super().__init__()
        self.a = a
        self.b = b
        self.diffusivity = diffusivity  # spatial mobility
        self.bc = bc  # boundary condition
        self.rng = rng

    def get_initial_state(self, grid):
        """Prepare a useful initial state."""
        u = pde.ScalarField(grid, self.a, label="Field $u$")
        v = self.b / self.a + 0.1 * pde.ScalarField.random_normal(grid, label="Field $v$", rng=self.rng)
        return pde.FieldCollection([u, v])

    def evolution_rate(self, state, t=0):
        """Pure python implementation of the PDE."""
        u, v = state
        u = (u+1)/2 * (mx_u-mn_u) + mn_u
        v = (v+1)/2 * (mx_v-mn_v) + mn_v
        scale_u = 2/(mx_u-mn_u)
        scale_v = 2/(mx_v-mn_v)
        rhs = state.copy()
        d0, d1 = self.diffusivity
        rhs[0] = d0 * u.laplace(self.bc) + self.a - (self.b + 1) * u + u**2 * v
        rhs[1] = d1 * v.laplace(self.bc) + self.b * u - u**2 * v
        rhs[0] = scale_u * rhs[0] * scal
        rhs[1] = scale_v * rhs[1] * scal
        return rhs

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        d0, d1 = self.diffusivity
        a, b = self.a, self.b
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(state_data, t):
            u = state_data[0]
            v = state_data[1]
            u = (u+1)/2 * (mx_u-mn_u) + mn_u
            v = (v+1)/2 * (mx_v-mn_v) + mn_v
            scale_u = 2/(mx_u-mn_u)
            scale_v = 2/(mx_v-mn_v)

            rate = np.empty_like(state_data)
            rate[0] = scale_u * (d0 * laplace(u) + a - (1 + b) * u + v * u**2) * scal
            rate[1] = scale_v * (d1 * laplace(v) + b * u - v * u**2) * scal
            return rate

        return pde_rhs
        
class CstmExplicitSolver(pde.ExplicitSolver):
    name = 'explicit_mine'
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _make_single_step_fixed_euler(self, state):
        if self.pde.is_sde:
            # handle stochastic version of the pde
            raise Exception('NOt implemented 1')

        else:
            # handle deterministic version of the pde
            rhs_pde = self._make_pde_rhs(state, backend=self.backend)

            def stepper(state_data: np.ndarray, t: float, dt) -> None:
                """perform a single Euler step"""
                state_data += dt * rhs_pde(state_data, t)

        return stepper

    def _make_single_step_rk45(self, state):
        if self.pde.is_sde:
            raise RuntimeError("Runge-Kutta stepper does not support stochasticity")

        # obtain functions determining how the PDE is evolved
        rhs = self._make_pde_rhs(state, backend=self.backend)

        def stepper(state_data: np.ndarray, t: float, dt) -> None:
            """compiled inner loop for speed"""
            # calculate the intermediate values in Runge-Kutta
            k1 = dt * rhs(state_data, t)
            k2 = dt * rhs(state_data + 0.5 * k1, t + 0.5 * dt)
            k3 = dt * rhs(state_data + 0.5 * k2, t + 0.5 * dt)
            k4 = dt * rhs(state_data + k3, t + dt)

            state_data += (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return stepper

    def _make_single_step_fixed_dt(self, state):
        
        if self.scheme == "euler":
            return self._make_single_step_fixed_euler(state)
        elif self.scheme in {"runge-kutta", "rk", "rk45"}:
            return self._make_single_step_rk45(state)
        else:
            raise ValueError(f"Explicit scheme `{self.scheme}` is not supported")
        
    def _make_fixed_stepper(self, state):
        single_step = self._make_single_step_fixed_dt(state)
        # print('before')

        if self._compiled:
            # print('compiling')
            sig_single_step = (nb.typeof(state.data), nb.double, nb.double)
            single_step = nb.jit(sig_single_step)(single_step)

        def fixed_stepper(
            state_data: np.ndarray, t_start: float, steps: int, dt
        ):
            """perform `steps` steps with fixed time steps"""
            modifications = 0.0
            for i in range(steps):
                # calculate the right hand side
                t = t_start + i * dt
                single_step(state_data, t, dt)

            return t + dt

        if self._compiled:
            sig_fixed = (nb.typeof(state.data), nb.double, nb.int_, nb.double)
            fixed_stepper = nb.jit(sig_fixed)(fixed_stepper)

        return fixed_stepper

    def make_stepper(self, state):
        fixed_stepper = self._make_fixed_stepper(state)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state, dt):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end, dt)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step, Gdt, Fdt):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        self.Gdt = Gdt
        self.Fdt = Fdt
        

    def run_G(self, t0, t1, u0):
        fields = [pde.ScalarField(self.grid, u0[i,...]) for i in range(u0.shape[0])]
        state = pde.FieldCollection(fields)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        fields = [pde.ScalarField(self.grid, u0[i,...]) for i in range(u0.shape[0])]
        state = pde.FieldCollection(fields)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'Brus'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return self.u0.shape
        


class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        if kwargs.get('_run_from_int', False):
            out = self._run_from_int(*args, **kwargs)
        else:
            out = self._run(*args, **kwargs)
        return out
    
    def _parareal(self, model, early_stop=None, parall='Serial', store_int=False, light=False, **kwargs):
        tspan, N, epsilon, n = self.tspan, self.N, self.epsilon, self.n
        u0 = self.u0
        solver: SolverAbstr = self.solver

        if kwargs.get('debug', False):
            print('WARNING: PararealLight does not support debug mode')
                         
        t = np.linspace(tspan[0], tspan[1], num=N+1)           
        I = 0                             
            
        parall = parall.lower()
        if parall == 'mpi':
            if 'pool' not in kwargs:
                raise Exception('MPI parallel backend requested but no pool of worker provided')
            pool = kwargs['pool']
            
        if 'verbose' in kwargs:
            verbose = kwargs['verbose']
        else:
            verbose = self.verbose

        if verbose and parall != 'serial':
            print(f'Running {model.name} with {parall} parallel backend')
            
        conv_int = []
        
        # u = np.empty((N+1, n, N+1))
        # uG = np.empty((N+1, n, N+1))
        # uF = np.empty((N+1, n, N+1))
        err = np.empty((N+1, N))
        # u.fill(np.nan)
        # uG.fill(np.nan)
        # uF.fill(np.nan)
        err.fill(np.nan)

        # err_old = np.empty((N+1, N))
        # err_old.fill(np.nan)

        u_curr = np.empty((N+1, *n))
        u_next = np.empty((N+1, *n))
        uG_curr = np.empty((N+1, *n))
        uG_next = np.empty((N+1, *n))
        uF_curr = np.empty((N+1, *n))
        uF_next = np.empty((N+1, *n))
        u_curr.fill(np.nan)
        u_next.fill(np.nan)
        uG_curr.fill(np.nan)
        uG_next.fill(np.nan)
        uF_curr.fill(np.nan)
        uF_next.fill(np.nan)
        
        # x_old = np.zeros((0, n))
        # D_old = np.zeros((0,n))
        x = np.zeros((0, np.prod(n)))
        D = np.zeros((0, np.prod(n)))
        
        G_time = 0
        F_time = 0
        F_time_serial = 0
        
            
        # u[0,:,:] = u0[:, np.newaxis]
        # uG[0,:,:] = u[0,:,:]
        # uF[0,:,:] = u[0,:,:]

        u_curr[0,...] = u0
        uG_curr[0,...] = u_curr[0,...]
        uF_curr[0,...] = u_curr[0,...]
        u_next[0,...] = u_curr[0,...]
        uG_next[0,...] = u_curr[0,...]
        uF_next[0,...] = u_curr[0,...]
        
        
        # Initialization: run G sequentially
        temp = u0
        for i in range(N):
            temp, temp_t = solver.run_G_timed(t[i], t[i+1], temp)
            G_time += temp_t
            # uG[i+1,:,0] = temp
            uG_curr[i+1,...] = temp

        del temp, temp_t
        # u[:,:,0] = uG[:,:,0]
        u_curr[:,...] = uG_curr[:,...]

        
        #Step 2: integrate using F (fine solver) in parallel with the current best initial
        # values
        for k in range(N):
            if verbose == 'v':
                print(f'{self.ode_name} {model.name} iteration number (out of {N}): {k+1} ')
                
            s_time = time.time()
            if parall == 'mpi':
                out = list(pool.map(solver.run_F_timed, t[I:N], t[I+1:N+1], [u_curr[i,...] for i in range(I,N)]))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
                del _temp_uFs
            elif parall == 'joblib':
                out = Parallel(-1)(delayed(lambda i: solver.run_F_timed(t[i], t[i+1], u_curr[i,...]))(i) for i in range(I,N))
                _temp_uFs = np.array([i[0] for i in out])
                uF_curr[I+1:N+1,...] = _temp_uFs
                F_time_serial += np.array([i[1] for i in out]).mean()
            else:
                temp_t = 0
                for i in range(I, N):
                    # temp_old = solver.run_F_timed(t[i], t[i+1], u[i,:,k])
                    temp, _temp_t_int = solver.run_F_timed(t[i], t[i+1], u_curr[i,...])
                    # uF[i+1,:,k] = temp_old[0]
                    uF_curr[i+1,...] = temp
                    temp_t += _temp_t_int
                F_time_serial += temp_t/(N-I)
            F_time += time.time() - s_time
            del s_time
            # save values forward (as solution at time I+1 is now converged)

            uG_next[I+1,...] = uG_curr[I+1,...]
            uF_next[I+1,...] = uF_curr[I+1,...]
            u_next[I+1,...] = uF_curr[I+1,...]
            I = I + 1
            # collect training data
            x_to_add = u_curr[I-1:N+1-1,...]
            D_to_add = uF_curr[I:N+1,...] - uG_curr[I:N+1,...]
            x = np.vstack([x, x_to_add.reshape(x_to_add.shape[0], np.prod(x_to_add.shape[1:]), order='C')])
            D = np.vstack([D, D_to_add.reshape(D_to_add.shape[0], np.prod(D_to_add.shape[1:]), order='C')])
            
            
            # early stop if only one interval was missing
            if I == N:
                if verbose == 'v':
                    print('WARNING: early stopping')
                err_old = np.nextafter(epsilon, 0)
                err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
                err[-1,k] = np.nextafter(epsilon, 0)
                break
            
            
            model.fit_timed(x, D, k=k)
                
            for i in range(I, N):

                

                # run G solver on best initial value
                temp, temp_t = solver.run_G_timed(t[i], t[i+1], u_next[i,...])
                G_time += temp_t
                uG_next[i+1,...] = temp
                del temp, temp_t
                
                mdl_inpt = u_next[i,...].reshape(1,-1, order='C')
                # preds = model.predict_timed(u_next[i,...].reshape(1,-1), 
                #                             uF_curr[i+1,...], uG_curr[i+1,...], i=i)

                _start_time_ = time.time()
                preds = model.predict_timed(mdl_inpt, 
                                            uF_curr[i+1,...], uG_curr[i+1,...], i=i)
                print(f'Done update {i-I} of {N-I} in {time.time()-_start_time_}s')
                preds = preds.reshape(*n, order='C')
                
                
                # do predictor-corrector update
                # u[i+1,:,k+1] = uF[i+1,:,k] + uG[i+1,:,k+1] - uG[i+1,:,k]
                u_next[i+1,...] = preds + uG_next[i+1,...]
                
            
            # error catch
            a = 0
            if np.any(np.isnan(uG_next[:,...])):
                raise Exception("NaN values in initial coarse solve - increase Ng!")
            # if np.any(np.isnan(uG[:,:, k+1])):
            #     raise Exception("NaN values in initial coarse solve - increase Ng!")
                           
            # Step 4: Converence check
            # checks whether difference between solutions at successive iterations
            # are small, if so then that time slice is considered converged. 
                          
            # err_old[:,k] = np.linalg.norm(u[:,:,k+1] - u[:,:,k], np.inf, 1)
            # err_old[I,k] = 0

            err[:,k] = np.array(list(map(lambda x: np.linalg.norm(x.ravel(), np.inf), (u_next - u_curr))))
            # err[:,k] = np.linalg.norm(u_next[:,...] - u_curr[:,...], np.inf, 1)
            err[I,k] = 0
            

            u_curr[...] = u_next[...]
            uG_curr[...] = uG_next[...]
            II = I
            for p in range(II+1, N+1):
                if err[p, k] < epsilon:
                    u_next[p,...] = u_curr[p,...]
                    uG_next[p,...] = uG_curr[p,...]
                    uF_next[p,...] = uF_curr[p,...]
                    I += 1
                else:
                    break
            uF_curr[...] = uF_next[...]


            if verbose == 'v':    
                print('--> Converged:', I)
            conv_int.append(I)
            if store_int:
                raise NotImplementedError('PararealLight does not support storing intermediate results')
                # name_base = f'{self.ode_name}_{self.N}_{model.name}_int'
                # int_dir = kwargs.get('int_dir', '')
                # name_base = kwargs.get('int_name', name_base)
                # int_name = f'{name_base}_{k}'
                # _objs = {'t':t, 'I':I, 'verbose':verbose,
                #      'u':u[...,:k+2], 'uG':uG[...,:k+2], 'uF':uF[...,:k+2], 'err':err[...,:k+2], 'x':x, 'D':D, 
                #      'G_time':G_time, 'F_time':F_time,
                #      'early_stop':early_stop, 'parall':parall, 'store_int':store_int, 'kwargs':kwargs,
                #      'k':k, 'conv_int': conv_int}
                
                # self.store(path=os.path.join(int_dir, name_base), name=int_name, mdl=model, objs=_objs)
                
                
            if I == N:
                break
            if (early_stop is not None) and k == (early_stop-1):
                if verbose == 'v':
                    print('Early stopping due to user condition.')
                break
        
        debug_dict = {}
            
            
        timings = {'F_time':F_time, 'G_time': G_time, 'F_time_serial_avg': F_time_serial/(k+1)}
        timings.update(model.get_times())
        
        # return {'t':t, 'u':u[:,:,:k+1], 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
        #         'u_curr':u_curr, 'uG_curr':uG_curr, 'uF_curr':uF_curr, 'err_old':err_old[:,:k+1],
        #         'uF':uF[:,:,:k+1], 'uG':uG[:,:,:k+1], 'x_old':x_old, 'D_old':D_old,
        #         'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
        #         'conv_int':conv_int}

        if light:
            if 'by_iter' in timings:
                timings.pop('by_iter')
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int}
        else:
            return {'t':t, 'u':u_curr, 'err':err[:, :k+1], 'x':x, 'D':D, 'k':k+1, 
                'timings':timings, 'debug_dict':debug_dict, 'converged':I==N, 
                'conv_int':conv_int}



rng = np.random.default_rng(2345)
eq = BrusselatorPDE(rng=rng)
grid = pde.UnitGrid([d]*space_size, periodic=[True]*space_size)  

state = eq.get_initial_state(grid)
init_cond = state.data
init_cond[0,...] = 2*(init_cond[0,...]-mn_u)/(mx_u-mn_u)-1
init_cond[1,...] = 2*(init_cond[1,...]-mn_v)/(mx_v-mn_v)-1

F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

F_fixed_stepper = F.make_stepper(pde.FieldCollection([pde.ScalarField(grid),pde.ScalarField(grid)]))
G_fixed_stepper = G.make_stepper(pde.FieldCollection([pde.ScalarField(grid),pde.ScalarField(grid)]))
def F_wrapped_stepper(state, t_start: float, t_end: float, Fdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps, Fdt)

def G_wrapped_stepper(state, t_start: float, t_end: float, Gdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps, Gdt) 


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)
    
    # print(N, mdl, dx, ode.name)
    
    
    ########## CHANGE THIS #########
    dir_name = 'BrusPDE'
    ################################
    
    name = f'{dir_name}_{d}_{mdl}'
    # assert workers >= N
    
    # generate folder
    if not os.path.exists(dir_name):
        os.mkdir(dir_name)
    
    
    ########## CHANGE THIS #########
        
    solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper, Gdt=Gdt, Fdt=Fdt)
    ode = KSODE(d, space_size, init_cond.copy()) 
    p = PararealLightMod(ode, solver, (0, T), N, epsilon=5e-7)

    # First run is biased, do one iter of Para
    res = p.run(model='elm', degree=1, m=4, pool=pool, parall='mpi', light=True, early_stop=1)
    
    #####################################
    
    # run the code, storing intermediates in custom folder
    if mdl == 'para':
        res = p.run(pool=pool, parall='mpi', light=True)
    elif mdl == 'elm':
        res = p.run(model='elm', degree=1, m=4, pool=pool, parall='mpi', light=True)
    elif mdl == 'nngp':
        res = p.run(model='nngp', pool=pool, parall='mpi', light=True, 
                    nn=20)
    else:
        raise Exception('Unknown model type', mdl)
        
    print(res['timings'])
        
    # dump the final result
    p.store(name=name, path=dir_name)

## Plots

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

def calc_t(sec, full=False):
    days = sec // (24 * 3600)
    hours = (sec - days * 24 * 3600) // 3600
    minutes = (sec - days * 24 * 3600 - hours * 3600) // 60
    seconds = sec - days * 24 * 3600 - hours * 3600 - minutes * 60

    if full:
        out = f'{int(days)}d {int(hours)}h {int(minutes)}m {int(seconds)}s'
    else:
        if days > 0:
            if minutes > 30:
                hours += 1
            out = f'{int(days)}d {int(hours)}h'
        else:
            if hours > 0:
                if seconds > 30:
                    minutes += 1
                out = f'{int(hours)}h {int(minutes)}m'
            elif minutes > 0:
                if seconds > 30:
                    minutes += 1
                out = f'{int(minutes)}m'
            else:
                out = f'{int(seconds)}s'
    return out

N = 512
# using 1e-7
F_oneint_avg = np.array([11.81, 42.32, 142.64, 284.23])*5
G_time_based_on_d_K = {
    32: {2: 0.5, 7: 0.83},
    64: {2: 0.83, 7:1.42},
    20: {2: 1.35, 8: 2.79},
    25: {2: 2.2, 9: 4.86}
}
iters = {
    'para': [7,7,8,9],
    'elm': [2,2,2,2],
    'nngp': [2, 2, 2, 2],
} 

mdl_cost = {
    'para': [0.023, 0.047, 0.10, 0.18],
    'elm': [2.82, 7.8, 13.26, 27.67],
    'nngp': [26.3*512*2, 503.84*512*2, np.nan, np.nan],
} 

# runtime
store_time = {}
for mdl in ['para', 'elm', 'nngp', 'fine']:
    for idx, d in enumerate([32, 64, 20, 25]):
        dct = store_time.get(mdl, {})
        if mdl == 'fine':
            dct[d] = N*F_oneint_avg[idx]
        else:
            dct[d] = (iters[mdl][idx]*F_oneint_avg[idx]+G_time_based_on_d_K[d][iters[mdl][idx]]+mdl_cost[mdl][idx])
        store_time[mdl] = dct

# speed-up
store = {}
for mdl in ['para', 'elm', 'nngp']:
    for idx, d in enumerate([32, 64, 20, 25]):
        dct = store.get(mdl, {})
        dct[d] = store_time['fine'][d]/store_time[mdl][d]
        store[mdl] = dct

# convert inner dictionary to list
store = {k: [v[i] for i in [32, 64, 20, 25]] for k, v in store.items()}
store_time = {k: [v[i] for i in [32, 64, 20, 25]] for k, v in store_time.items()}

# # runtime pretty
# store_time_pretty = {}
# for mdl in ['para', 'elm', 'nngp', 'fine']:
#     for idx, d in enumerate([32, 64, 20, 25]):
#         dct = store_time_pretty.get(mdl, {})
#         dct[d] = calc_t(store_time[mdl][d])
#         store_time_pretty[mdl] = dct
# # store pretty -> round values
# store_pretty = {}
# for mdl in ['para', 'elm', 'nngp']:
#     for idx, d in enumerate([32, 64, 20, 25]):
#         dct = store_pretty.get(mdl, {})
#         dct[d] = round(store[mdl][d], 2)
#         store_pretty[mdl] = dct

# for i in store_time_pretty:
#     print(i, store_time_pretty[i])
# for i in store_pretty:
    # print(i, store_pretty[i])



fontsize = 15
fs_ticks = 13
ds = np.array([2*32**2, 2*64**2, 2*20**3, 2*25**3])
Ns = [2,2, 3,3]
c = {'elm':'red','para':'gray','nngp':'blue', 'fine':'black'}    
legend_tags = {'elm':'RandNet-Parareal', 'para':'Parareal', 'nngp':'nnGParareal', 'fine':'Fine solver'}
markers = {'elm':'2', 'para':'x', 'nngp':'+', 'fine':'_'}
fig, axs = plt.subplots(1,2,figsize = [6.4*2+1, 4.8])
ax=axs[0]
# fig, ax = plt.subplots()
for key, val in store.items():
    mdl = key
    x = np.log10(ds)
    y = np.array(val)  
    if mdl == 'nngp':
        ax.scatter(x, y, label=legend_tags[mdl], marker=markers[mdl], facecolors='blue', edgecolor='blue', s=300)
        # facecolor='none'
    else:
        ax.scatter(x, y, label=legend_tags[mdl], c=c[mdl], marker=markers[mdl], s=300)

    ax.set_xticks(x)
ax.scatter(x, np.array([1]*len(x)),  c='black',marker='_', label='Fine solver',s=300)
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(ax.get_xticks())
ax2.set_xticklabels(np.array(Ns))
ax.tick_params(axis='both', labelsize=fs_ticks)
ax2.tick_params(axis='both', labelsize=fs_ticks)
ax2.set_xlabel(r"Spatial dimensions", fontsize=fontsize)
# ax.legend(prop={'size':fontsize}, loc='upper left', frameon=False, fontsize=fontsize)
ax.set_xlabel(r'$\log_{10}(d)$',fontsize=fontsize)
ax.set_ylabel('Speed-up',fontsize=fontsize)
fig.tight_layout()

ax = axs[1]
for key, val in store_time.items():
    # mdl, _ = key.split('_')
    mdl = key
    x = np.log10(ds)
    y = np.array(val)  
    y = np.log10(y/3600)
    if mdl == 'nngp':
        ax.scatter(x, y, label=legend_tags[mdl], marker=markers[mdl], facecolors='blue', edgecolor='blue', s=300)
        # facecolor='none'
    else:
        ax.scatter(x, y, label=legend_tags[mdl], c=c[mdl], marker=markers[mdl], s=300)

    ax.set_xticks(x)
# ax.scatter(x, np.array([1]*len(x)),  c='black',marker='_', label='Fine solver',s=300)
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(ax.get_xticks())
ax2.set_xticklabels(np.array(Ns))
ax.tick_params(axis='both', labelsize=fs_ticks)
ax2.tick_params(axis='both', labelsize=fs_ticks)
ax2.set_xlabel(r"Spatial dimensions", fontsize=fontsize)
# ax.legend(prop={'size':fontsize}, loc='upper left', frameon=False, fontsize=fontsize)
ax.legend(prop={'size':fontsize}, loc='upper left', bbox_to_anchor=(1,1), frameon=False, fontsize=fontsize)
ax.set_xlabel(r'$\log_{10}(d)$',fontsize=fontsize)
ax.set_ylabel(r'Runtime (hours, $\log_{10}$)',fontsize=fontsize)
fig.suptitle(r'2D and 3D Brussellator PDE', fontsize=fontsize+2)
fig.tight_layout()


name = 'Brus_PDE_speedup_w_time'
fig.savefig(os.path.join('revision', name), bbox_inches='tight')
fig.savefig(os.path.join('revision', name+'.pdf'), bbox_inches='tight')



# Kuramoto-shivashinski

In [ ]:
#%%
from mpi4py.futures import MPIPoolExecutor
import os
import sys

print('starting')


from parareal import PararealLight
from solver import SolverAbstr
import pde
import numpy as np
import matplotlib.pyplot as plt
from systems import ODE
import time

import numba as nb


import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pde 
import numpy as np
import numba as nb
from pde.solvers.base import SolverBase
from systems import ODE
from solver import SolverAbstr

scale=1
d = 32
space_size = 2


mn, mx = -400, 2.5

class KuramotoSivashinskyPDENorm(pde.PDEBase):
    """Implementation of the normalized Kuramoto–Sivashinsky equation."""

    def __init__(self, bc="auto_periodic_neumann"):
        super().__init__()
        self.bc = bc

    def evolution_rate(self, state, t=0):
        """Implement the python version of the evolution equation."""

        # un-normalize state
        state = (state+1)/2 * (mx-mn) + mn

        # make original computation
        state_lap = state.laplace(bc=self.bc)
        state_lap2 = state_lap.laplace(bc=self.bc)
        state_grad_sq = state.gradient_squared(bc=self.bc)
        out = -state_grad_sq / 2 - state_lap - state_lap2
        # mn = -400
        # mx = 400
        # rescale output
        scale = 2/(mx-mn)
        return scale * out

    def _make_pde_rhs_numba(self, state):
        """Nunmba-compiled implementation of the PDE."""
        
        gradient_squared = state.grid.make_operator("gradient_squared", bc=self.bc)
        laplace = state.grid.make_operator("laplace", bc=self.bc)

        @nb.njit
        def pde_rhs(data, t):
            data = (data+1)/2 * (mx-mn) + mn
            out =  -0.5 * gradient_squared(data) - laplace(data + laplace(data))
            # mn = -400
            # mx = 400
            scale = 2/(mx-mn)
            return scale * out

        return pde_rhs
        
class CstmExplicitSolver(pde.ExplicitSolver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _make_single_step_fixed_euler(self, state):
        if self.pde.is_sde:
            # handle stochastic version of the pde
            raise Exception('NOt implemented 1')

        else:
            # handle deterministic version of the pde
            rhs_pde = self._make_pde_rhs(state, backend=self.backend)

            def stepper(state_data: np.ndarray, t: float, dt) -> None:
                """perform a single Euler step"""
                state_data += dt * rhs_pde(state_data, t)

        return stepper

    def _make_single_step_rk45(self, state):
        if self.pde.is_sde:
            raise RuntimeError("Runge-Kutta stepper does not support stochasticity")

        # obtain functions determining how the PDE is evolved
        rhs = self._make_pde_rhs(state, backend=self.backend)

        def stepper(state_data: np.ndarray, t: float, dt) -> None:
            """compiled inner loop for speed"""
            # calculate the intermediate values in Runge-Kutta
            k1 = dt * rhs(state_data, t)
            k2 = dt * rhs(state_data + 0.5 * k1, t + 0.5 * dt)
            k3 = dt * rhs(state_data + 0.5 * k2, t + 0.5 * dt)
            k4 = dt * rhs(state_data + k3, t + dt)

            state_data += (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return stepper

    def _make_single_step_fixed_dt(self, state):
        
        if self.scheme == "euler":
            return self._make_single_step_fixed_euler(state)
        elif self.scheme in {"runge-kutta", "rk", "rk45"}:
            return self._make_single_step_rk45(state)
        else:
            raise ValueError(f"Explicit scheme `{self.scheme}` is not supported")
        
    def _make_fixed_stepper(self, state):
        single_step = self._make_single_step_fixed_dt(state)
        # print('before')

        if self._compiled:
            # print('compiling')
            sig_single_step = (nb.typeof(state.data), nb.double, nb.double)
            single_step = nb.jit(sig_single_step)(single_step)

        def fixed_stepper(
            state_data: np.ndarray, t_start: float, steps: int, dt
        ):
            """perform `steps` steps with fixed time steps"""
            modifications = 0.0
            for i in range(steps):
                # calculate the right hand side
                t = t_start + i * dt
                single_step(state_data, t, dt)

            return t + dt

        if self._compiled:
            sig_fixed = (nb.typeof(state.data), nb.double, nb.int_, nb.double)
            fixed_stepper = nb.jit(sig_fixed)(fixed_stepper)

        return fixed_stepper

    def make_stepper(self, state):
        fixed_stepper = self._make_fixed_stepper(state)
        return fixed_stepper

class CstmController():

    def __init__(self, t_range, stepper, tracker=None):
        self.wrapped_stepper = stepper
        self.t_range = t_range

    @property
    def t_range(self) -> tuple[float, float]:
        return self._t_range

    @t_range.setter
    def t_range(self, value):
        try:
            self._t_range: tuple[float, float] = (0, float(value))  # type: ignore
        except TypeError:  # assume a single number was given
            if len(value) == 2:  # type: ignore
                self._t_range = tuple(value)  # type: ignore
            else:
                raise ValueError(
                    "t_range must be set to a single number or a tuple of two numbers"
                )

    def run(self, initial_state, dt):
        state = initial_state.copy()
        t_start, t_end = self.t_range

        # initialize the stepper
        self.wrapped_stepper(state, t_start, t_end, dt)

        if not np.all(np.isfinite(state.data)):
            raise Exception("Field was not finite")
        return state
    
    


class KSSolver(SolverAbstr):
    def __init__(self, grid, G_step, F_step, Gdt, Fdt):
        self.grid=grid

        self.g_stepper = G_step
        self.f_stepper = F_step
        self.Gdt = Gdt
        self.Fdt = Fdt
        

    def run_G(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.g_stepper)
        return controller.run(state, dt=self.Gdt).data
    
    def run_F(self, t0, t1, u0):
        state = pde.ScalarField(self.grid, u0)
        controller = CstmController(t_range=(t0,t1), stepper=self.f_stepper)
        return controller.run(state, dt=self.Fdt).data
    
class KSODE(ODE):
    def __init__(self, d, space_size, u0):
        self.name = 'KS'
        self.d = d
        self.space_size = space_size
        self.u0 = u0

    def get_init_cond(self):
        return self.u0
    
    def get_dim(self):
        return (self.d,)*self.space_size
        


class PararealLightMod(PararealLight):
    def run(self, *args, **kwargs):
        pool = self._get_pool(*args, **kwargs)
        kwargs['pool'] = pool
        if kwargs.get('_run_from_int', False):
            out = self._run_from_int(*args, **kwargs)
        else:
            out = self._run(*args, **kwargs)
        return out





rng = np.random.default_rng(2345)
eq = KuramotoSivashinskyPDENorm()
grid = pde.UnitGrid([d]*space_size)  
F = CstmExplicitSolver(eq, scheme='rk', backend='numba')
G = CstmExplicitSolver(eq, scheme='euler', backend='numba')

init_cond_state = pde.ScalarField.random_uniform(grid, rng=rng)
init_cond = init_cond_state.data
# normalize initial condition
init_cond = 2*(init_cond-mn)/(mx-mn)-1

F_fixed_stepper = F.make_stepper(pde.ScalarField(grid))
G_fixed_stepper = G.make_stepper(pde.ScalarField(grid))
def F_wrapped_stepper(state, t_start: float, t_end: float, Fdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Fdt)))
    return F_fixed_stepper(state.data, t_start, steps, Fdt)

def G_wrapped_stepper(state, t_start: float, t_end: float, Gdt) -> float:
    steps = max(1, int(np.ceil((t_end - t_start) / Gdt)))
    return G_fixed_stepper(state.data, t_start, steps, Gdt)

#%

import json

def store_l(l):
    try:
        with open('revision_KS.txt', 'a') as f:
            f.write(json.dumps(l)+'\n')
    except Exception as e:
        pass

def do(pool, inp):
    Gdt, T = inp
    Fdt = 0.001
    solver = KSSolver(grid, G_step=G_wrapped_stepper, F_step=F_wrapped_stepper, Gdt=Gdt, Fdt=Fdt)

    

    ode = KSODE(d, space_size, init_cond.copy()) 
    N = 235
    p = PararealLightMod(ode, solver, (0, T), N, epsilon=5e-7)

    l = [scale, d, space_size, Gdt, T, N, Fdt]
    try:
        res_elm = p.run(model='elm', degree=1, m=4,pool=pool, parall='mpi', light=True)
        l.extend(('elm', res_elm['k']))
    except Exception as e:
        l.extend(('elm', -1, str(e)))
        print(l)
        store_l(l)  
        return

    try:
        res = p.run(pool=pool, parall='mpi', light=True)
        l.extend(['para', res['k'], res_elm['timings'], res['timings']])
        print(l)
        store_l(l)
    except Exception as e:
        l.extend(['para', -1, str(e), res_elm['timings']])
        store_l(l)
        print(l)
        return
        


if __name__ == '__main__':
    
    avail_work = int(os.getenv('SLURM_NTASKS'))
    workers = avail_work - 1
    print('Total workes', workers)
    pool = MPIPoolExecutor(workers)

    from itertools import product, repeat

    T = [20, 21, 22,25, 30, 35, 40, 45, 50, 55, 60,61,62, 65,70, 100,101,102,105, 150, 155, 200,201,202, 250, 300]
    Gdt = 10**(np.linspace(0,-3, 50))
    args = list(product(Gdt, T))

    print(len(args))

    np.random.seed(34)
    args = np.random.permutation(args)

    out = list(map(do, repeat(pool), args))


# from IPython.display import clear_output
# from matplotlib import pyplot as plt

# l = [init_cond.copy()]
# for i in range(20):
#     l.append(solver.run_F(i,i+1,l[-1]))
#     plt.imshow(l[-1][0,...])
#     plt.title(i)
#     clear_output(wait=True)
#     plt.pause(0.1)

# Plot of computational time

In [ ]:
#%%
# 
import numpy as np
import matplotlib.pyplot as plt

# First Figure
m = 20
k = [16, 20, 50, 100, 100, 100, 100]
N = [64, 128, 256, 512, 512, 512, 512]
d = [722, 1568, 3362, 11856, 25538, 53792, 110450]

T = []
for i in range(7):
    T.append(k[i] * N[i] * max(d[i] / N[i], 1) * (m**2.8 + m**2 + d[i] * (m**2 + 2 * m) + m * np.log(N[i] * k[i])))

M = 100
k = [12, 12, 12, 6, 5, 4, 4]
T1 = []
n = 4
for i in range(7):
    T1.append(k[i] * N[i] * (M**2.8 + M**2 * n + d[i] * (M**2 + 3 * M * n) + n * np.log(N[i] * k[i])) / N[i])



fontsize = 16
fs_ticks = 13


fig, axs = plt.subplots(1,2,figsize=(20,5))
ax1 = axs[0]
ax1.plot(np.log10(48) * np.ones(7), label='48 hours threshold', color="red")
ax1.plot(np.log10(np.array(T1) / 3600 / 600000 / 30), '-o',label='RandNet-Parareal', color="blue", markersize=5)
ax1.plot(np.log10(np.array(T) / 3600 / 600000), '-o', label='nnGParareal', color="gray", markersize=5)
ax1.set_xlabel('Figure A', fontsize=fontsize+7, labelpad=20)
ax1.set_ylabel(f'$log_{10}(T)$ hours', fontsize=fontsize)

row1 = ['d=722', 'd=1568', 'd=3362', 'd=11856', 'd=25538', 'd=53792', 'd=110450']
row2 = ['N=64', 'N=128', 'N=256', 'N=512', 'N=512', 'N=512', 'N=512']
tickLabels = [f'{r1}\n{r2}' for r1, r2 in zip(row1, row2)]
ax1.set_xticks(range(len(d)), tickLabels)
ax1.set_yticks(np.arange(-3, 7))
ax1.tick_params(axis='both', labelsize=fs_ticks)
ax1.legend(loc='best', fontsize=fontsize)
ax1.set_title('Model computational cost', fontsize=fontsize+2)

#%

# Second Figure
f = [2*60, 2*60, 3*60, 8*60, 23*60, 60*60, 2*3600+37*60]


f_nngp = []
for i in range(7):
    # T[i] = k[i] * N[i] * max(d[i] / N[i], 1) * (m**2.8 + m**2 + d[i] * (m**2 + 2 * m) + m * np.log(N[i] * k[i]))
    f_nngp.append(k[i] * f[i])

# T1 = []
f_randNet = []
for i in range(7):
    # T1.append(k[i] * N[i] * (M**2.8 + M**2 * n + d[i] * (M**2 + 3 * M * n) + n * np.log(N[i] * k[i])) / N[i])
    f_randNet.append(k[i] * f[i])


ax2 = axs[1]
ax2.plot(np.log10(48) * np.ones(7), label='48 hours threshold', color="red")
ax2.plot(np.log10(np.array(f_randNet) / 3600 + np.array(T1) / 3600 / 600000 / 30), '-o', label='RandNet-Parareal', color="blue", markersize=5)
ax2.plot(np.log10(np.array(f_nngp) / 3600 + np.array(T) / 3600 / 600000), '-o', label='nnGParareal$', color="gray", markersize=5)
ax2.set_xlabel('Figure B', fontsize=fontsize+7, labelpad=20)
ax2.set_ylabel(f'$log_{10}(T)$ hours', fontsize=fontsize)
ax2.set_yticks(np.arange(-1, 7))
ax2.set_xticks(range(len(d)), tickLabels)
ax2.legend(loc='best', fontsize=fontsize)
ax2.tick_params(axis='both', labelsize=fs_ticks)
ax2.set_title(r'Model computational cost with fine solver cost $T(\mathcal{F})$', fontsize=fontsize+2)
fig.tight_layout()

fig.savefig('model_computational_cost.pdf')
